# ✨ 24 · Denoising without clean data: J-invariance, cycle consistency and score matching
### Classical filters, Noise2Self, a blind-spot network trained on ONE noisy tomogram, DUAL's unpaired cycle, and score-based denoising with target guidance

**BiologyToResearch** · Part E — Unsupervised & geometric deep learning (2020–2026) · Module 24 of 34

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/cryoem/tutorial/BiologyToResearch/notebooks/24_denoising_without_clean_data.ipynb)

| | |
|---|---|
| **Hardware target** | Google Colab free tier — CPU is enough (≈ 4–6 min to run everything); a GPU is optional |
| **Core frameworks** | Python 3, NumPy, SciPy, Matplotlib, scikit-image, PyTorch |
| **Prerequisites** | Module 05 (PyTorch), Module 06 (noise, CTF), Module 07 (missing wedge), Module 09 (in silico cell), Module 10 (FSC) |
| **Estimated time** | 150 min (reading + running + the ✏️ exercises) |
| **Lab papers covered** | **J-invariant volume shuffle** (Liu, Kassab, Xu & Ho, WACV 2025); **DUAL** (Zeng *et al.*, bioRxiv 2024); **Score-based matching with target guidance** (ECCV 2026; taught from the released AITom code — paper text not verified); optional: **MicroFM** (Zhan *et al.*, CVPR 2026, fluorescence microscopy) — see [`PAPER_ATLAS.md`](../PAPER_ATLAS.md) |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis.
> All experiments are small, controlled *toy versions* of published methods (a 48³ simulated tomogram, 24×24 particle
> images, networks with a few thousand weights, seconds of training). They illustrate the core ideas; they do **not**
> reproduce the papers' numbers. **Denoised images are interpretations, not data** — Section 7 shows how a denoiser's prior
> can hide real structure or invent new structure. Always validate conclusions on the raw data.

## 🎯 Learning objectives

By the end of this notebook you will be able to:

1. **Apply** classical filters (Gaussian, median, bilateral, anisotropic diffusion) to a simulated tomogram and **measure**
   PSNR and SSIM — and explain why these metrics need a ground truth that real data never have.
2. **Explain** J-invariance and why, for a J-invariant denoiser, the self-supervised loss $\lVert f(x)-x\rVert^2$ equals the
   true loss plus a constant; **tune** a filter without any clean data (Noise2Self) or with two noisy halves (Noise2Noise).
3. **Build** a 3D blind-spot network (centrally masked 5³ convolution + volume unshuffle with factor 3), **verify**
   numerically that every output voxel ignores its own input, **train** it on one noisy volume, and show that an ordinary
   network collapses to the identity.
4. **Measure** denoising without ground truth with even/odd FSC — and prove to yourself that a linear filter cannot change it.
5. **Explain** DUAL's unpaired, cycle-consistent simulation + denoising, and show in a toy why a cycle loss alone does not
   make a realistic simulator.
6. **Explain** the score, Tweedie's formula and denoising score matching; **train** a score network on noisy particle images,
   add target guidance from a bank of clean references (as in the AITom TSM code), and **explain** how priors make
   denoisers hallucinate.

## 🗺️ Contents

| Section | What you will do | ⏱ |
|---|---|---|
| 0 · Setup | toolkit, a simulated cell with an answer key | 5 min |
| 1 · Classical filters | four filters, PSNR/SSIM | 15 min |
| 2 · J-invariance | Noise2Self and Noise2Noise: tuning without clean data | 20 min |
| 3 · J-invariant volume shuffle | blind-spot check, train on one volume | 30 min |
| 4 · Validation without ground truth | even/odd FSC | 10 min |
| 5 · DUAL | unpaired cycle: simulate and denoise | 20 min |
| 6 · Score matching | Tweedie, DSM, target guidance | 30 min |
| 7 · Hallucination | priors that snap and references that reproduce themselves | 10 min |
| ✅ / 🧠 / 🚀 | checkpoint, discussion, going further | 10 min |

## 🧫 The biological question

A cryo-electron tomogram is a 3D picture of a cell, but it is taken with so few electrons (to avoid destroying the
sample) that the signal-to-noise ratio of every voxel is far below 1. Biologists still need to *look* at tomograms — to
trace membranes, spot filaments, choose particles, judge whether a drug changed a structure — and many downstream
programs (pickers, segmenters) also work better on cleaner input.

The obvious recipe — "train a network on pairs of noisy and clean tomograms" — is impossible: **no clean tomogram of a
real cell exists**. This notebook is about the clever ways around that: networks that learn to denoise from the noisy
data alone (J-invariance), from two unrelated sets of data (cycle consistency), or by learning *what clean data look
like* statistically (score matching).

---
## 0 · Environment setup

In [ ]:
# ── Hardware check ──
import platform
import subprocess
import sys

print("Python", sys.version.split()[0], "on", platform.system())
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True, timeout=10).stdout.strip()
    print("GPU:", gpu or "none detected")
except (FileNotFoundError, subprocess.TimeoutExpired):
    print("GPU: none detected — that is fine, this notebook runs on a CPU.")
try:
    import torch
    print("PyTorch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
except ImportError:
    print("PyTorch not installed yet (the next cell installs it).")

In [ ]:
# ── Dependencies (Colab already has these; we install only what is missing) ──
import importlib

for module, pip_name in [("numpy", "numpy"), ("scipy", "scipy"), ("matplotlib", "matplotlib"),
                         ("skimage", "scikit-image"), ("torch", "torch")]:
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)
print("✅ All dependencies are available.")

In [ ]:
# ── Global imports and configuration ──
import os
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy import ndimage
from skimage.metrics import structural_similarity
from skimage.restoration import denoise_bilateral

SEED = 0
rng = np.random.default_rng(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
plt.rcParams.update({"figure.dpi": 90, "figure.figsize": (6.4, 3.6), "axes.titlesize": 10,
                     "axes.labelsize": 9, "image.cmap": "gray"})
print("NumPy", np.__version__, "| PyTorch", torch.__version__, "| device:", DEVICE, "| seed", SEED)

### 🧰 Course toolkit

Verbatim copies from [`course_toolkit.py`](../course_toolkit.py): zoo building (Module 00), rotations (Module 02),
Fourier masks (Module 03), noise (Module 04), CTF (Module 06), missing wedge (Module 07), the in silico cell (Module 09)
and FSC (Module 10).

In [ ]:
# ── Course toolkit (identical to ../course_toolkit.py v2.0; copied so this notebook runs alone) ──
import struct

import numpy as np
from scipy import ndimage

ZOO = {
    "4v4r": dict(name="70S ribosome", organism="bacterial (E. coli / T. thermophilus)", mass_kDa=2330, symmetry="C1"),
    "4v7r": dict(name="80S ribosome", organism="S. cerevisiae", mass_kDa=3330, symmetry="C1"),
    "5mrc": dict(name="mitochondrial ribosome", organism="S. cerevisiae", mass_kDa=3320, symmetry="C1"),
    "3j9i": dict(name="20S proteasome", organism="Thermoplasma acidophilum", mass_kDa=710, symmetry="D7"),
    "6utj": dict(name="singly capped proteasome", organism="Thermoplasma acidophilum", mass_kDa=910, symmetry="C7"),
    "4cr2": dict(name="26S proteasome", organism="S. cerevisiae", mass_kDa=1280, symmetry="C1"),
    "4v94": dict(name="TRiC/CCT chaperonin (×2)", organism="S. cerevisiae", mass_kDa=1910, symmetry="C2"),
    "1bxn": dict(name="RuBisCO (L4S4 half)", organism="Cupriavidus necator", mass_kDa=290, symmetry="C4"),
    "1qvr": dict(name="ClpB (3 of 6 subunits)", organism="Thermus thermophilus", mass_kDa=290, symmetry="C1"),
    "3cf3": dict(name="p97/VCP ATPase", organism="Mus musculus", mass_kDa=570, symmetry="C6"),
    "2cg9": dict(name="Hsp90–Sba1 chaperone", organism="S. cerevisiae", mass_kDa=190, symmetry="C2"),
    "1u6g": dict(name="Cand1–Cul1–Roc1 ligase", organism="H. sapiens", mass_kDa=260, symmetry="C1"),
    "3d2f": dict(name="Sse1p–Hsp70 chaperone", organism="S. cerevisiae / H. sapiens", mass_kDa=230, symmetry="C1"),
    "1s3x": dict(name="Hsp70 ATPase domain", organism="H. sapiens", mass_kDa=44, symmetry="C1"),
}
_BOND = dict(N_CA=1.458, CA_C=1.525, C_N=1.329, C_O=1.231)
_ANGLE = dict(N_CA_C=111.2, CA_C_N=116.2, C_N_CA=121.7, CA_C_O=120.5)
_SIDECHAIN = {
    "GLY": [], "ALA": [], "SER": [("OG", "O")], "CYS": [("SG", "S")], "VAL": [("CG1", "C"), ("CG2", "C")],
    "THR": [("OG1", "O"), ("CG2", "C")], "PRO": [("CG", "C"), ("CD", "C")],
    "LEU": [("CG", "C"), ("CD1", "C"), ("CD2", "C")], "ILE": [("CG1", "C"), ("CG2", "C"), ("CD1", "C")],
    "ASP": [("CG", "C"), ("OD1", "O"), ("OD2", "O")], "ASN": [("CG", "C"), ("OD1", "O"), ("ND2", "N")],
    "GLU": [("CG", "C"), ("CD", "C"), ("OE1", "O"), ("OE2", "O")], "GLN": [("CG", "C"), ("CD", "C"), ("OE1", "O"), ("NE2", "N")],
    "LYS": [("CG", "C"), ("CD", "C"), ("CE", "C"), ("NZ", "N")], "MET": [("CG", "C"), ("SD", "S"), ("CE", "C")],
    "ARG": [("CG", "C"), ("CD", "C"), ("NE", "N"), ("CZ", "C"), ("NH1", "N"), ("NH2", "N")],
    "HIS": [("CG", "C"), ("ND1", "N"), ("CD2", "C"), ("CE1", "C"), ("NE2", "N")],
    "PHE": [("CG", "C"), ("CD1", "C"), ("CD2", "C"), ("CE1", "C"), ("CE2", "C"), ("CZ", "C")],
    "TYR": [("CG", "C"), ("CD1", "C"), ("CD2", "C"), ("CE1", "C"), ("CE2", "C"), ("CZ", "C"), ("OH", "O")],
    "TRP": [("CG", "C"), ("CD1", "C"), ("CD2", "C"), ("NE1", "N"), ("CE2", "C"), ("CE3", "C"), ("CZ2", "C"),
            ("CZ3", "C"), ("CH2", "C")],
}
_AA_FREQ = {"LEU": 9.9, "ALA": 8.3, "GLY": 7.1, "VAL": 6.9, "GLU": 6.8, "SER": 6.6, "ILE": 5.9, "LYS": 5.8, "ARG": 5.5,
            "ASP": 5.5, "THR": 5.3, "PRO": 4.7, "ASN": 4.1, "GLN": 3.9, "PHE": 3.9, "TYR": 2.9, "MET": 2.4, "HIS": 2.3,
            "CYS": 1.4, "TRP": 1.1}
def _nerf(a, b, c, bond, angle, torsion):
    """Place atom d so that |cd| = bond, angle(b,c,d) = angle and dihedral(a,b,c,d) = torsion (degrees)."""
    bc = (c - b) / np.linalg.norm(c - b)
    n = np.cross(b - a, bc)
    n /= np.linalg.norm(n)
    m = np.cross(n, bc)
    ang, tor = np.radians(angle), np.radians(torsion)
    d = np.array([-bond * np.cos(ang), bond * np.sin(ang) * np.cos(tor), bond * np.sin(ang) * np.sin(tor)])
    return c + d[0] * bc + d[1] * m + d[2] * n


def build_backbone(phi, psi, omega=None) -> np.ndarray:
    """Backbone atoms N, CA, C, O, CB (array n×5×3, Å) of a chain with the given torsion angles (degrees)."""
    n = len(phi)
    omega = np.full(n, 180.0) if omega is None else np.asarray(omega, float)
    N = [np.zeros(3)]
    CA = [np.array([_BOND["N_CA"], 0.0, 0.0])]
    C = [_nerf(np.array([0.0, 1.0, 0.0]), N[0], CA[0], _BOND["CA_C"], _ANGLE["N_CA_C"], phi[0])]
    for i in range(n - 1):
        N.append(_nerf(N[i], CA[i], C[i], _BOND["C_N"], _ANGLE["CA_C_N"], psi[i]))
        CA.append(_nerf(CA[i], C[i], N[i + 1], _BOND["N_CA"], _ANGLE["C_N_CA"], omega[i]))
        C.append(_nerf(C[i], N[i + 1], CA[i + 1], _BOND["CA_C"], _ANGLE["N_CA_C"], phi[i + 1]))
    N, CA, C = np.array(N), np.array(CA), np.array(C)
    O = np.array([_nerf(N[i], CA[i], C[i], _BOND["C_O"], _ANGLE["CA_C_O"], psi[i] + 180.0) for i in range(n)])
    b, c = CA - N, C - CA
    a = np.cross(b, c)
    CB = -0.58273431 * a + 0.56802827 * b - 0.54067466 * c + CA      # ideal Cβ position
    return np.stack([N, CA, C, O, CB], 1)


def _ideal_helix_template(n=160):
    """An ideal α-helix (φ = −57.8°, ψ = −47.0°) with its axis along +z through x = y = 0."""
    bb = build_backbone(np.full(n, -57.8), np.full(n, -47.0))
    ca = bb[:, 1]
    centre = ca.mean(0)
    u = np.linalg.svd(ca - centre)[2][0]
    if np.dot(ca[-1] - ca[0], u) < 0:
        u = -u
    # rotation taking u to z
    v = np.cross(u, [0, 0, 1.0]); s = np.linalg.norm(v); cth = u[2]
    K = np.array([[0, -v[2], v[1]], [v[2], 0, -v[0]], [-v[1], v[0], 0]])
    Rm = np.eye(3) + K + K @ K * ((1 - cth) / (s ** 2 + 1e-12))
    xyz = (bb - centre) @ Rm.T
    axis_xy = xyz[:, 1, :2].mean(0)
    xyz[..., :2] -= axis_xy
    return xyz


_HELIX = _ideal_helix_template()
def _sidechains(bb, resnames, rng):
    """Approximate side-chain heavy atoms grown outward from Cβ (positions are schematic; counts/elements are right)."""
    names, elems, xyz, resid = [], [], [], []
    ca, cb = bb[:, 1], bb[:, 4]
    out = cb - ca
    out /= np.linalg.norm(out, axis=1, keepdims=True)
    for i, rn in enumerate(resnames):
        sc = _SIDECHAIN[rn]
        if not sc:
            continue
        perp = np.cross(out[i], rng.normal(size=3)); perp /= np.linalg.norm(perp)
        for k, (an, el) in enumerate(sc):
            depth = 1.0 + (k // 2)                               # schematic zig-zag outward
            pos = cb[i] + 1.5 * depth * out[i] + 0.9 * (1 if k % 2 else -1) * perp
            names.append(an); elems.append(el); xyz.append(pos); resid.append(i)
    return names, elems, np.array(xyz).reshape(-1, 3), np.array(resid, int)


def _loop_path(a, b, outward, step=3.8):
    """Cα positions of a loop from Cα `a` to Cα `b`: a circular arc bulging along `outward`, cut into equal chords so
    that every Cα–Cα distance (including the two ends) is exactly `step` Å."""
    c = np.linalg.norm(b - a)
    e = (b - a) / c
    u = outward - (outward @ e) * e                                  # bulge direction ⟂ to the chord
    if np.linalg.norm(u) < 1e-6:
        u = np.cross(e, [0.0, 0.0, 1.0]) if abs(e[2]) < 0.9 else np.cross(e, [1.0, 0.0, 0.0])
    u = u / np.linalg.norm(u)
    m = max(int(np.ceil(c / step)), 1)                               # m new residues → n = m + 1 equal chords
    n = m + 1
    lo, hi = 1e-9, 2 * np.pi - 1e-9                                  # arc angle Θ: sin(Θ/2) / sin(Θ/2n) = c / step
    for _ in range(60):
        th = (lo + hi) / 2
        lo, hi = (th, hi) if np.sin(th / 2) / np.sin(th / (2 * n)) > c / step else (lo, th)
    R = step / (2 * np.sin(th / (2 * n)))
    centre = (a + b) / 2 - R * np.cos(th / 2) * u
    beta = np.pi / 2 + th / 2 - th * np.arange(1, n) / n             # angles of the new points, a → b over the bulge
    return centre + R * (np.outer(np.cos(beta), e) + np.outer(np.sin(beta), u))


def _assemble_chain(ca_path_segments, rng, chain="A", start_resnum=1):
    """Turn helices (n×5×3 arrays) joined by loops into one chain with residue names and side chains."""
    blocks = []
    for k, seg in enumerate(ca_path_segments):
        blocks.append(seg)
        if k + 1 < len(ca_path_segments):
            a, b = seg[-1, 1], ca_path_segments[k + 1][0, 1]
            path = _loop_path(a, b, (a + b) / 2 - seg.reshape(-1, 3).mean(0))
            m = len(path)
            tang = np.gradient(path, axis=0) if m > 1 else np.array([b - a])
            tang /= np.linalg.norm(tang, axis=1, keepdims=True)
            side = np.cross(tang, rng.normal(size=3)); side /= np.linalg.norm(side, axis=1, keepdims=True)
            loop = np.stack([path - 1.2 * tang, path, path + 1.2 * tang, path + 1.2 * tang + 1.0 * side,
                             path - 1.3 * side], 1)
            blocks.append(loop)
    bb = np.concatenate(blocks, 0)
    aa = list(_AA_FREQ)
    p = np.array([_AA_FREQ[a] for a in aa]); p /= p.sum()
    resnames = rng.choice(aa, len(bb), p=p)
    bb_names = ["N", "CA", "C", "O", "CB"]
    names, elems, xyz, resid = [], [], [], []
    for i in range(len(bb)):
        for j, an in enumerate(bb_names):
            if an == "CB" and resnames[i] == "GLY":
                continue
            names.append(an); elems.append(an[0]); xyz.append(bb[i, j]); resid.append(i)
    sn, se, sx, sr = _sidechains(bb, resnames, rng)
    names += sn; elems += se; resid = np.concatenate([np.array(resid, int), sr])
    xyz = np.concatenate([np.array(xyz), sx], 0)
    order = np.argsort(resid, kind="stable")
    return dict(xyz=xyz[order].astype(np.float32), name=np.array(names)[order], element=np.array(elems)[order],
                resname=resnames[resid[order]], resnum=(resid[order] + start_resnum).astype(int),
                chain=np.full(len(order), chain))


def helix_bundle(radii_A, rng, chain="A", spacing=10.0):
    """A globular 'domain' that fills an ellipsoid (semi-axes in Å, long axis = local z) with antiparallel helices."""
    a, b, c = radii_A[0] + 2.5, radii_A[1] + 2.5, radii_A[2] + 1.5   # helices stop short of the surface: pad
    rows = []
    j = 0
    while (j * spacing * np.sqrt(3) / 2) <= b:
        rows.append(j); rows.append(-j) if j else None; j += 1
    pts = []
    for jj in sorted(set(rows)):
        y = jj * spacing * np.sqrt(3) / 2
        xs = np.arange(-a, a + 1e-6, spacing) + (spacing / 2 if jj % 2 else 0)
        row = [(x, y) for x in xs if (x / a) ** 2 + (y / b) ** 2 < 1]
        pts.append(row if (jj % 2 == 0) else row[::-1])
    pts = [p for row in pts for p in row]
    segs = []
    for k, (x, y) in enumerate(pts):
        h = c * np.sqrt(max(0.0, 1 - (x / a) ** 2 - (y / b) ** 2)) - 2.0
        n = int(2 * h / 1.5)
        if n < 6:
            continue
        seg = _HELIX[:n].copy()
        seg[..., 2] -= seg[:, 1, 2].mean()
        phase = rng.uniform(0, 2 * np.pi)
        Rz = np.array([[np.cos(phase), -np.sin(phase), 0], [np.sin(phase), np.cos(phase), 0], [0, 0, 1]])
        seg = seg @ Rz.T
        if len(segs) % 2 == 1:                                   # antiparallel neighbours: 180° turn about x
            seg = seg * np.array([1.0, -1.0, -1.0])
        seg = seg + np.array([x, y, rng.uniform(-1.0, 1.0)])
        segs.append(seg)
    return _assemble_chain(segs, rng, chain)


def _transform(atoms, R=np.eye(3), t=(0, 0, 0)):
    out = dict(atoms)
    out["xyz"] = (atoms["xyz"] @ np.asarray(R, float).T + np.asarray(t, float)).astype(np.float32)
    return out


def merge_atoms(parts):
    keys = parts[0].keys()
    return {k: np.concatenate([p[k] for p in parts]) for k in keys}


def _rz(a):
    return np.array([[np.cos(a), -np.sin(a), 0], [np.sin(a), np.cos(a), 0], [0, 0, 1.0]])


def _rx(a):
    return np.array([[1, 0, 0], [0, np.cos(a), -np.sin(a)], [0, np.sin(a), np.cos(a)]])


def _ry(a):
    return np.array([[np.cos(a), 0, np.sin(a)], [0, 1, 0], [-np.sin(a), 0, np.cos(a)]])


_CHAIN_IDS = "ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz0123456789"
def _ring(n, radius, z, radii, phase=0.0, tilt=None):
    """n identical subunits (semi-axes `radii` nm: radial, tangential, axial) around the z axis."""
    out = []
    for k in range(n):
        a = phase + 2 * np.pi * k / n
        R = _rz(a) if tilt is None else _rz(a) @ tilt
        out.append(dict(radii=radii, centre=(radius * np.cos(a), radius * np.sin(a), z), R=R, key=("ring", radii, tilt is None)))
    return out


def _lobe(radii, centre, R=None):
    return [dict(radii=radii, centre=centre, R=np.eye(3) if R is None else R, key=None)]


def _dihedral(subunits):
    """Add the partners generated by a 2-fold axis along x (dihedral symmetry)."""
    Rx = _rx(np.pi)
    return subunits + [dict(radii=s["radii"], centre=tuple(Rx @ np.array(s["centre"])), R=Rx @ s["R"], key=s["key"])
                       for s in subunits]


def _blueprint(pid):
    """Subunit layout (nm) of the code-generated stand-in for each zoo entry."""
    if pid in ("4v4r", "4v7r", "5mrc"):
        s = {"4v4r": 1.0, "4v7r": 1.13, "5mrc": 1.14}[pid]
        L = [_lobe((9.0 * s, 8.0 * s, 6.5 * s), (0, 0, 4.5 * s)),                       # large subunit body
             _lobe((3.0 * s, 3.0 * s, 3.0 * s), (0, 1.0 * s, 11.0 * s)),                # central protuberance
             _lobe((2.0 * s, 2.0 * s, 3.5 * s), (-9.0 * s, 0, 7.0 * s), _ry(-0.6)),     # L1-like stalk
             _lobe((1.4 * s, 1.4 * s, 4.0 * s), (9.5 * s, 0, 6.5 * s), _ry(0.7)),       # L7/L12-like stalk
             _lobe((7.5 * s, 5.0 * s, 3.8 * s), (1.0 * s, 0, -4.5 * s)),                # small subunit body
             _lobe((3.5 * s, 3.2 * s, 3.0 * s), (-5.5 * s, -1.0 * s, -6.5 * s))]        # small subunit head
        if pid == "4v7r":                                                                # eukaryotic expansion segments
            L += [_lobe((3.2, 2.6, 2.8), (8.0, 7.0, 7.0)), _lobe((3.0, 2.6, 3.2), (-6.0, -8.0, -6.5)),
                  _lobe((2.4, 2.4, 3.6), (-10.5, 4.0, -1.0), _ry(0.5))]
        if pid == "5mrc":                                                                # mitochondria-specific proteins
            L += [_lobe((4.2, 3.6, 2.8), (0, -7.5, 12.5)), _lobe((3.4, 3.0, 3.4), (9.0, -6.0, -8.0)),
                  _lobe((3.0, 3.0, 3.0), (-9.5, 7.0, 1.0)), _lobe((2.5, 4.5, 2.0), (2.0, 9.0, 9.0))]
        prng = np.random.default_rng(_seed(pid) + 1)                                     # rRNA-like surface bumps
        for _ in range(8):
            d = prng.normal(size=3); d /= np.linalg.norm(d)
            L += [_lobe(tuple(prng.uniform(1.2, 1.9, 3)), tuple(s * d * np.array([9.5, 8.0, 10.5])), _rz(prng.uniform(0, 6)))]
        return [x for l in L for x in l]
    if pid in ("3j9i", "6utj", "4cr2"):
        core = []
        for z, radii, ph in [(1.9, (1.9, 1.7, 2.0), 0.0), (5.6, (2.1, 1.7, 1.85), np.pi / 7)]:
            core += _ring(7, 3.6, z, radii, ph)
        core = _dihedral(core)                                                           # α7β7β7α7, D7
        if pid == "6utj":
            core += _ring(7, 3.3, 10.0, (1.9, 1.6, 2.4), np.pi / 14)                     # one heptameric cap: C7
        if pid == "4cr2":
            core += _ring(6, 4.3, 9.6, (2.2, 1.8, 2.0), 0.3)                             # base ring of six
            core += (_lobe((4.0, 3.0, 3.0), (3.5, 0, 14.5)) + _lobe((3.0, 2.5, 3.5), (-1.5, 3.0, 15.5))
                     + _lobe((2.5, 2.5, 2.5), (-3.5, -2.0, 13.5)) + _lobe((3.0, 3.0, 2.5), (-4.5, 0.5, 10.5)))
        return core
    if pid == "4v94":
        one = []
        for z in (3.6, -3.6):
            one += _ring(8, 5.2, z, (2.6, 2.0, 3.4), np.pi / 8 if z < 0 else 0.0)
        two = []
        for dx, rot in [(-8.5, 0.0), (8.5, np.pi)]:
            for s in one:
                c = _rz(rot) @ np.array(s["centre"])
                two.append(dict(radii=s["radii"], centre=(c[0] + dx, c[1], c[2]), R=_rz(rot) @ s["R"], key=s["key"]))
        return two
    if pid == "1bxn":
        # like the deposited file (the crystal's asymmetric unit): one L4S4 layer = half of the L8S8 enzyme, C4.
        # `_dihedral` of this list would give the complete enzyme (D4).
        return _ring(4, 3.6, 2.2, (2.6, 2.4, 2.4)) + _ring(4, 2.6, 5.0, (1.9, 1.7, 1.35), np.pi / 4)
    if pid == "1qvr":
        # like the deposited file: three neighbouring subunits (half of the C6 hexameric ring), C1
        tiers = [_ring(6, 4.2, 2.2, (2.2, 2.0, 2.0)), _ring(6, 3.9, -2.2, (2.3, 2.0, 2.0), np.pi / 6),
                 _ring(6, 7.2, 2.4, (0.9, 0.9, 4.6), 0.4, tilt=_rx(1.2)),               # coiled-coil 'propellers'
                 _ring(6, 3.8, 6.6, (1.7, 1.6, 1.8), 0.5)]                               # N-domains on top
        return [s for t in tiers for s in t[:3]]
    if pid == "3cf3":
        return (_ring(6, 4.0, 2.0, (2.1, 2.0, 2.0)) + _ring(6, 4.0, -2.2, (2.2, 2.0, 2.1), np.pi / 6)
                + _ring(6, 6.6, 3.0, (2.0, 1.6, 1.8), 0.25))                             # N-domains flush: C6
    if pid == "2cg9":
        half = (_lobe((1.9, 1.7, 1.7), (2.2, 0.8, 6.0)) + _lobe((2.2, 1.8, 2.0), (2.6, -0.4, 2.0))
                + _lobe((1.7, 1.5, 1.8), (1.4, 0.3, -2.2)) + _lobe((1.4, 1.2, 1.2), (4.2, 1.6, 7.6)))
        Rz = _rz(np.pi)
        return half + [dict(radii=h["radii"], centre=tuple(Rz @ np.array(h["centre"])), R=Rz @ h["R"], key=None) for h in half]
    if pid == "1u6g":
        out = []
        for k in range(6):                                                                # curved cullin stalk
            t = k / 5
            out += _lobe((1.4, 1.4, 1.6), (8.0 * np.sin(0.9 * t) - 3.0, 0, 9.0 * t - 4.5), _ry(0.9 * t))
        out += _lobe((2.2, 2.0, 2.0), (4.0, 0, 5.8)) + _lobe((1.2, 1.1, 1.1), (6.0, 1.8, 6.5))
        for k in range(10):                                                               # Cand1-like superhelix
            a = 1.6 * np.pi * k / 9
            out += _lobe((1.5, 1.4, 1.4), (-3.0 + 7.0 * np.sin(0.9 * k / 9) + 2.6 * np.cos(a), 2.6 * np.sin(a), 9.0 * k / 9 - 4.5))
        return out
    if pid == "3d2f":
        comp = (_lobe((1.8, 1.6, 1.8), (0, 0, 0)) + _lobe((1.8, 1.6, 1.8), (0, 3.2, 0)) + _lobe((1.6, 1.4, 2.4), (3.0, 1.6, 1.8))
                + _lobe((1.6, 1.5, 1.6), (-3.2, 1.0, -1.0)) + _lobe((1.6, 1.5, 1.6), (-3.2, 3.6, -0.6)))
        R2, t2 = _rx(2.3) @ _rz(0.7), np.array([1.5, -4.5, 5.5])
        return comp + [dict(radii=c["radii"], centre=tuple(R2 @ np.array(c["centre"]) + t2), R=R2 @ c["R"], key=None) for c in comp]
    if pid == "1s3x":
        return (_lobe((1.4, 1.4, 1.6), (-1.6, 0, -1.2)) + _lobe((1.3, 1.3, 1.5), (-1.4, 0.2, 1.6))
                + _lobe((1.4, 1.4, 1.6), (1.6, 0, -1.2)) + _lobe((1.2, 1.3, 1.4), (1.5, -0.3, 1.8)))
    raise KeyError(pid)


def _seed(pid):
    return int.from_bytes(pid.encode(), "little") % (2 ** 31)


_SCALE = {"4v4r": 0.923, "4v7r": 0.906, "5mrc": 0.883, "3j9i": 0.930, "6utj": 0.930, "4cr2": 0.930, "4v94": 0.886,
          "1bxn": 0.930, "1qvr": 0.885, "3cf3": 0.935, "2cg9": 0.995, "1u6g": 0.934, "3d2f": 0.997, "1s3x": 0.884}
_ATOMS_CACHE = {}
def zoo_atoms(pid):
    """Atomic model (dict of arrays) of the code-generated stand-in for zoo entry `pid` (Å, centred at the origin)."""
    pid = pid.lower()
    if pid in _ATOMS_CACHE:
        return _ATOMS_CACHE[pid]
    rng = np.random.default_rng(_seed(pid))
    built, parts = {}, []
    f = _SCALE.get(pid, 1.0)
    for k, s in enumerate(_blueprint(pid)):
        radii_A = tuple(10.0 * f * r for r in s["radii"])
        key = s["key"]
        if key is not None and key in built:
            sub = built[key]
        else:
            sub = helix_bundle(radii_A, rng)
            if key is not None:
                built[key] = sub
        part = _transform(sub, s["R"], 10.0 * f * np.array(s["centre"]))
        part["chain"] = np.full(len(part["xyz"]), _CHAIN_IDS[k % len(_CHAIN_IDS)])
        parts.append(part)
    atoms = merge_atoms(parts)
    if pid in ("3j9i", "6utj", "4cr2"):                             # lay the proteasome barrels along x (side view down z)
        atoms["xyz"] = (atoms["xyz"] @ _ry(np.pi / 2).T).astype(np.float32)
    atoms["xyz"] -= atoms["xyz"].mean(0)
    _ATOMS_CACHE[pid] = atoms
    return atoms


_Z = {"C": 6.0, "N": 7.0, "O": 8.0, "S": 16.0, "P": 15.0, "H": 1.0}
def atoms_to_map(xyz, weights=None, voxel=10.0, box=None, sigma_A=None):
    """Density map [z, y, x] from atom positions (Å): trilinear splatting on a grid twice as fine, Gaussian blur
    (σ = `sigma_A`, default = voxel/2), then 2×2×2 binning to `voxel` Å. Centred on the box if `box` is given."""
    xyz = np.asarray(xyz, float)
    w = np.ones(len(xyz)) if weights is None else np.asarray(weights, float)
    fine = voxel / 2.0
    if box is None:
        ext = np.ptp(xyz, 0).max() + 6 * voxel
        box = int(np.ceil(ext / voxel)) + 1
    n = 2 * box
    centre = (n - 1) / 2.0
    p = xyz[:, ::-1] / fine + centre                                  # (z, y, x) grid coordinates
    grid = np.zeros((n, n, n), np.float32)
    i0 = np.floor(p).astype(int)
    f = p - i0
    for dz in (0, 1):
        for dy in (0, 1):
            for dx in (0, 1):
                idx = i0 + np.array([dz, dy, dx])
                wt = w * np.where(dz, f[:, 0], 1 - f[:, 0]) * np.where(dy, f[:, 1], 1 - f[:, 1]) * np.where(dx, f[:, 2], 1 - f[:, 2])
                ok = np.all((idx >= 0) & (idx < n), 1)
                np.add.at(grid, tuple(idx[ok].T), wt[ok])
    grid = ndimage.gaussian_filter(grid, (sigma_A if sigma_A else voxel / 2.0) / fine)
    return grid.reshape(box, 2, box, 2, box, 2).mean((1, 3, 5)).astype(np.float32)


def zoo_map(pid, voxel="10A"):
    """(density[z, y, x], voxel size Å, source) of the code-generated stand-in for a zoo entry (no download)."""
    atoms = zoo_atoms(pid)
    v = float(str(voxel).rstrip("A"))
    w = np.array([_Z.get(e, 6.0) for e in atoms["element"]])
    vol = atoms_to_map(atoms["xyz"], w, voxel=v)
    return vol, v, f"code-generated stand-in modelled on PDB {pid.upper()}"


def prepare_map(vol: np.ndarray, size: int = 32, sigma: float = 1.0) -> np.ndarray:
    """Low-pass, centre (by centre of mass), crop/pad to ``size``³ and scale to [0, 1].

    Zoo maps are sharp sums of atoms; a Gaussian blur of ~1 voxel (1 nm)
    mimics the resolution of a cellular tomogram.
    """
    v = ndimage.gaussian_filter(np.clip(vol, 0, None).astype(np.float32), sigma)
    s = np.array(v.shape)
    src0 = np.maximum((s - size) // 2, 0)                  # where the crop starts in the input ...
    dst0 = np.maximum((size - s) // 2, 0)                  # ... and where the pad starts in the output
    n = np.minimum(s, size)
    com = np.array(ndimage.center_of_mass(v))
    shift = (size - 1) / 2 - dst0 + src0 - com             # puts the centre of mass exactly at the output's centre
    v = ndimage.shift(v, shift, order=1)
    out = np.zeros((size, size, size), np.float32)
    out[dst0[0]:dst0[0] + n[0], dst0[1]:dst0[1] + n[1], dst0[2]:dst0[2] + n[2]] = \
        v[src0[0]:src0[0] + n[0], src0[1]:src0[1] + n[1], src0[2]:src0[2] + n[2]]
    out -= out.min()
    return out / (out.max() + 1e-12)


def load_zoo(ids=("4v4r", "3j9i", "6utj", "1bxn"), size: int = 32, sigma: float = 1.0,
             verbose: bool = True) -> dict:
    """Build and prepare several zoo maps. Returns {pdb_id: volume(size³)}."""
    zoo = {}
    for pid in ids:
        raw, vs, src = zoo_map(pid)
        zoo[pid] = prepare_map(raw, size=size, sigma=sigma)
        if verbose:
            print(f"  {pid.upper():5s} {ZOO.get(pid, {}).get('name', ''):28s} raw {raw.shape} @ {vs:.0f} Å/voxel  ← {src}")
    return zoo


def random_rotation(rng: np.random.Generator) -> np.ndarray:
    """Uniformly random proper rotation (det = +1) from a random unit quaternion."""
    q = rng.normal(size=4)
    q /= np.linalg.norm(q)
    w, x, y, z = q
    return np.array([
        [1 - 2 * (y * y + z * z), 2 * (x * y - z * w), 2 * (x * z + y * w)],
        [2 * (x * y + z * w), 1 - 2 * (x * x + z * z), 2 * (y * z - x * w)],
        [2 * (x * z - y * w), 2 * (y * z + x * w), 1 - 2 * (x * x + y * y)]])


_P = np.array([[0, 0, 1], [0, 1, 0], [1, 0, 0]], float)  # (x,y,z) <-> (z,y,x)
def rotate_volume(vol: np.ndarray, R: np.ndarray, shift_xyz=(0, 0, 0), order: int = 1,
                  cval: float = 0.0) -> np.ndarray:
    """Rotate a volume about its centre by R (then translate by shift, in voxels, (x,y,z)).

    out(p) = vol(Rᵀ (p − c − t) + c): the *object* is rotated by R and moved by t.
    """
    R_idx = _P @ R @ _P                       # same rotation written in (z, y, x) index order
    c = (np.array(vol.shape) - 1) / 2.0
    t_idx = _P @ np.asarray(shift_xyz, float)
    M = R_idx.T                               # inverse rotation maps output -> input
    offset = c - M @ (c + t_idx)
    return ndimage.affine_transform(vol, M, offset=offset, order=order, mode="constant", cval=cval)


def freq_grid(shape):
    """Frequency coordinates (cycles/voxel) for an unshifted FFT; returns list per axis."""
    return np.meshgrid(*[np.fft.fftfreq(n) for n in shape], indexing="ij")


def radial_frequency(shape) -> np.ndarray:
    """|k| in cycles/voxel for every coefficient of an unshifted FFT."""
    return np.sqrt(sum(k**2 for k in freq_grid(shape)))


def missing_wedge_mask(shape, tilt_max_deg: float = 60.0) -> np.ndarray:
    """Binary Fourier mask of a single-axis tilt series about y (±tilt_max).

    A coefficient (k_z, k_y, k_x) is measured if |k_z| ≤ |k_x|·tan(tilt_max)
    — the formula M(ξ) = 1(|ξ_3| ≤ |ξ_1| tan θ) used by Xu et al. (2012).
    For 2D images [z, x] pass a 2-tuple shape.
    """
    k = freq_grid(shape)
    kz, kx = k[0], k[-1]
    m = np.abs(kz) <= np.abs(kx) * np.tan(np.radians(tilt_max_deg)) + 1e-9
    m[(0,) * len(shape)] = True               # keep the mean (DC) term
    return m.astype(np.float32)


def apply_fourier_mask(vol: np.ndarray, mask: np.ndarray) -> np.ndarray:
    """Multiply the FFT of ``vol`` by ``mask`` (unshifted layout) and return the real part."""
    return np.real(np.fft.ifftn(np.fft.fftn(vol) * mask)).astype(np.float32)


def pearson(a: np.ndarray, b: np.ndarray) -> float:
    """Pearson correlation of two arrays (flattened)."""
    a = a.ravel() - a.mean()
    b = b.ravel() - b.mean()
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-12))


def electron_wavelength_A(kV: float = 300.0) -> float:
    """Relativistic electron wavelength in Å (0.0197 Å at 300 kV)."""
    V = kV * 1e3
    return 12.2643 / np.sqrt(V * (1 + 0.978466e-6 * V))


def ctf_2d(shape, pixel_A: float, defocus_um: float = 3.0, kV: float = 300.0,
           Cs_mm: float = 2.7, amp_contrast: float = 0.07, B: float = 0.0) -> np.ndarray:
    """Contrast transfer function on an unshifted FFT grid (underfocus is positive).

    CTF(k) = −[√(1−A²)·sin χ(k) + A·cos χ(k)]·exp(−B k²/4),
    χ(k) = π λ Δf k² − (π/2) C_s λ³ k⁴, with k in 1/Å.
    """
    lam = electron_wavelength_A(kV)
    k2 = sum(kk**2 for kk in freq_grid(shape)) / pixel_A**2
    dz = defocus_um * 1e4
    cs = Cs_mm * 1e7
    chi = np.pi * lam * dz * k2 - 0.5 * np.pi * cs * lam**3 * k2**2
    ctf = -(np.sqrt(1 - amp_contrast**2) * np.sin(chi) + amp_contrast * np.cos(chi))
    return (ctf * np.exp(-B * k2 / 4)).astype(np.float32)


def fsc(v1: np.ndarray, v2: np.ndarray, pixel_A: float = 10.0, n_shells: int | None = None):
    """Fourier shell (ring, in 2D) correlation. Returns (frequency_1/Å, fsc, n_coeffs)."""
    F1, F2 = np.fft.fftn(v1), np.fft.fftn(v2)
    k = radial_frequency(v1.shape)
    n_shells = n_shells or min(v1.shape) // 2
    idx = np.minimum((k * 2 * n_shells).astype(int), n_shells)
    num = np.bincount(idx.ravel(), (F1 * np.conj(F2)).real.ravel(), n_shells + 1)
    d1 = np.bincount(idx.ravel(), (np.abs(F1)**2).ravel(), n_shells + 1)
    d2 = np.bincount(idx.ravel(), (np.abs(F2)**2).ravel(), n_shells + 1)
    cnt = np.bincount(idx.ravel(), minlength=n_shells + 1)
    curve = num / np.sqrt(d1 * d2 + 1e-20)
    freq = (np.arange(n_shells + 1) + 0.5) / (2 * n_shells) / pixel_A
    return freq[:n_shells], curve[:n_shells], cnt[:n_shells]


def resolution_at(freq, curve, threshold: float = 0.143) -> float:
    """First crossing of the FSC curve below ``threshold`` → resolution in Å."""
    below = np.where(curve[1:] < threshold)[0]
    if len(below) == 0:
        return float(1 / freq[-1])
    i = below[0] + 1
    f0, f1, c0, c1 = freq[i - 1], freq[i], curve[i - 1], curve[i]
    f = f0 + (threshold - c0) * (f1 - f0) / (c1 - c0 + 1e-12)
    return float(1 / f)


def support_radius(t: np.ndarray, level: float = 0.1) -> float:
    """Radius (voxels) of the smallest centred sphere containing all voxels above ``level``·max."""
    pts = np.argwhere(t > level * t.max())
    if len(pts) == 0:
        return t.shape[0] / 2
    c = (np.array(t.shape) - 1) / 2
    return float(np.linalg.norm(pts - c, axis=1).max()) + 1.0


def make_crowded_cell(templates: dict, counts: dict, shape=(48, 160, 160), rng=None,
                      membrane: bool = True, tries_per_particle: int = 400):
    """Pack randomly rotated templates into a volume without overlap (bounding spheres).

    A simple rejection-sampling stand-in for the packing used by Pei et al. (2016)
    and Liu et al. (2020).  Particles are placed in random class order so that every
    class gets a fair chance.  Returns (clean_volume, particles) where each particle
    is dict(cls, center_zyx, R).  Optionally adds a spherical membrane vesicle.
    """
    rng = rng or np.random.default_rng(0)
    vol = np.zeros(shape, np.float32)
    occ_c, occ_r, particles = [], [], []
    if membrane:  # a vesicle: thin shell, ~4 nm thick lipid bilayer at 1 nm/voxel
        zc, yc, xc = shape[0] / 2, shape[1] * 0.78, shape[2] * 0.22
        rad = min(shape[1], shape[2]) * 0.16
        g = np.meshgrid(*[np.arange(n) for n in shape], indexing="ij")
        d = np.sqrt((g[0] - zc)**2 + (g[1] - yc)**2 + (g[2] - xc)**2)
        vol += 0.6 * np.exp(-((d - rad)**2) / (2 * 1.5**2))
        occ_c.append(np.array([zc, yc, xc])); occ_r.append(rad + 3)
    radii = {cls: support_radius(templates[cls], level=0.35) for cls in counts}
    order = [cls for cls, n in counts.items() for _ in range(n)]
    rng.shuffle(order)
    for cls in order:
        t, rad_t = templates[cls], radii[cls]
        for _ in range(tries_per_particle):
            c = np.array([rng.uniform(rad_t, s - rad_t) if s > 2 * rad_t else s / 2 for s in shape])
            if any(np.linalg.norm(c - oc) < rad_t + orr for oc, orr in zip(occ_c, occ_r)):
                continue
            R = random_rotation(rng)
            sub = rotate_volume(t, R)
            h = np.array(sub.shape) // 2
            lo = np.round(c).astype(int) - h
            sl_v, sl_s = [], []
            for a in range(3):
                a0, a1 = max(lo[a], 0), min(lo[a] + sub.shape[a], shape[a])
                sl_v.append(slice(a0, a1)); sl_s.append(slice(a0 - lo[a], a1 - lo[a]))
            vol[tuple(sl_v)] += sub[tuple(sl_s)]
            occ_c.append(c); occ_r.append(rad_t)
            particles.append(dict(cls=cls, center_zyx=c, R=R))
            break
    return vol, particles
print("🧰 toolkit loaded")

### Data: a small in silico cell — with an answer key

We pack ribosomes, proteasomes, capped proteasomes and RuBisCO into a 48 × 96 × 96 nm box with a membrane vesicle
(Module 09), then "image" it exactly like the toolkit's `simulate_tomogram`: CTF (defocus 5 µm) → missing wedge (±60°) →
noise. The difference: we **keep the noise-free tomogram** as an answer key, which real experiments never give you.

To imitate how real data are collected, we simulate **two independent half-dose tomograms**, *even* and *odd* (e.g.
from even- and odd-numbered movie frames); each has twice the noise variance of their average, the *full-dose*
tomogram. Full-dose SNR = 0.3.

In [ ]:
# ── Build the cell and its tomograms (full dose, even half, odd half, noise-free answer key) ──
t0 = time.time()
CAST = ["4v4r", "3j9i", "6utj", "1bxn"]
templates = load_zoo(CAST, size=32)
counts = {"4v4r": 4, "3j9i": 6, "6utj": 4, "1bxn": 10}
clean, particles = make_crowded_cell(templates, counts, shape=(48, 96, 96), rng=np.random.default_rng(SEED))
ctf = ctf_2d(clean.shape, pixel_A=10.0, defocus_um=5.0)
noise_free = apply_fourier_mask(apply_fourier_mask(clean, -ctf), missing_wedge_mask(clean.shape, 60))
SNR = 0.3                                             # full-dose SNR (variance ratio)
sigma_full = np.sqrt(noise_free.var() / SNR)
g_noise = np.random.default_rng(SEED + 1)
even = noise_free + np.sqrt(2) * sigma_full * g_noise.standard_normal(clean.shape).astype(np.float32)
odd = noise_free + np.sqrt(2) * sigma_full * g_noise.standard_normal(clean.shape).astype(np.float32)
full = 0.5 * (even + odd)                             # noise variance σ² (each half has 2σ²)

# one common affine normalisation (from the full-dose data) for everything, so errors stay comparable
MU, SD = full.mean(), full.std()
norm = lambda v: ((v - MU) / SD).astype(np.float32)
truth, noisy, even_n, odd_n = norm(noise_free), norm(full), norm(even), norm(odd)
RANGE = float(truth.max() - truth.min())
NOISE_VAR = float((sigma_full / SD) ** 2)             # noise variance in normalised units (known in simulation only)


def psnr(estimate, reference=None):
    """Peak signal-to-noise ratio in dB: 10·log10(range² / MSE), with the range of the noise-free answer key."""
    reference = truth if reference is None else reference
    return float(10 * np.log10(RANGE ** 2 / np.mean((estimate - reference) ** 2)))


print(f"{len(particles)} particles placed in {time.time() - t0:.1f} s; volume {noisy.shape}; "
      f"noise variance (normalised) = {NOISE_VAR:.3f}; raw PSNR = {psnr(noisy):.2f} dB")

---
## 📍 Where we are

Module 06 explained where cryo-EM noise comes from (few electrons), Module 07 the missing wedge, Module 10 how to measure
resolution with two independent halves. Module 23's Harmony produced clean pictures as a by-product of disentangling. Here
**denoising is the goal**, and the central difficulty is that we must learn it **without clean examples**.

---
## 1 · Classical filters and how we score them

| filter | idea | what it assumes |
|---|---|---|
| **Gaussian** | replace each voxel by a weighted average of its neighbourhood | signal is smooth |
| **median** | replace each voxel by the median of its neighbourhood | noise has outliers; edges matter |
| **bilateral** | average only neighbours with *similar intensity* | edges separate regions of different intensity |
| **anisotropic diffusion** (Perona–Malik) | let intensity "flow" between voxels, but slow the flow across strong gradients | edges = large gradients |

Two standard scores compare an estimate $\hat{y}$ with the true image $y$ (here: the noise-free tomogram):

$$\text{PSNR} = 10\log_{10}\frac{R^2}{\tfrac{1}{N}\sum_i(\hat y_i - y_i)^2}\ \text{dB},\qquad
\text{SSIM} = \text{local agreement of mean, contrast and structure (1 = identical).}$$

*In words:* PSNR is the error measured in decibels relative to the intensity range $R$ (+3 dB ≈ half the squared error);
SSIM asks whether local patterns look alike. **Both need the clean image** — fine in simulation, impossible in reality.
Note that our "clean" target is the noise-free *tomogram*: CTF and missing wedge are still in it, because a denoiser
removes noise, not the microscope's blur.

In [ ]:
# ── 1.1 Four classical filters on one tomogram slice ──
def perona_malik(img, n_iter=15, kappa=1.0, step=0.2):
    """Anisotropic diffusion (Perona–Malik, 2D): smooth, but conduct less across large gradients (edges)."""
    u = img.astype(np.float64).copy()
    for _ in range(n_iter):
        flux = 0.0
        for axis in (0, 1):
            for shift in (1, -1):
                d = np.roll(u, shift, axis=axis) - u                  # difference to one neighbour
                flux = flux + np.exp(-(d / kappa) ** 2) * d           # conductance falls at strong edges
        u += step * flux
    return u.astype(np.float32)


def bilateral(img, sigma_color=0.15, sigma_spatial=1.5):
    """Bilateral filter (scikit-image); intensities are rescaled to [0, 1] first, then mapped back."""
    lo, hi = img.min(), img.max()
    out = denoise_bilateral((img - lo) / (hi - lo), sigma_color=sigma_color, sigma_spatial=sigma_spatial)
    return (out * (hi - lo) + lo).astype(np.float32)


Z = noisy.shape[0] // 2
sl_noisy, sl_truth = noisy[Z], truth[Z]
results = {"noisy": sl_noisy,
           "Gaussian σ=1": ndimage.gaussian_filter(sl_noisy, 1.0),
           "median 3×3": ndimage.median_filter(sl_noisy, 3),
           "bilateral": bilateral(sl_noisy),
           "anisotropic diffusion": perona_malik(sl_noisy)}
fig, axes = plt.subplots(1, 6, figsize=(12, 2.5))
axes[0].imshow(sl_truth); axes[0].set_title("noise-free (answer key)", fontsize=8)
for ax, (name, im) in zip(axes[1:], results.items()):
    ssim = structural_similarity(sl_truth, im, data_range=RANGE)
    ax.imshow(im); ax.set_title(f"{name}\nPSNR {psnr(im, sl_truth):.1f} dB · SSIM {ssim:.2f}", fontsize=8)
for ax in axes:
    ax.axis("off")
plt.tight_layout(); plt.show()

🔍 **What just happened?** Every filter trades noise for blur. With default-ish settings they all raise PSNR by several
dB, and the edge-aware filters keep membrane and particle boundaries a little crisper. But notice *how* we chose their
settings: by peeking at the answer key. With a real tomogram you cannot compute PSNR at all — so how would you choose
σ? Section 2 answers this.

✏️ **Your turn.** Change `kappa` in `perona_malik` (try 0.3 and 3) and `sigma_color` in `bilateral` (0.05, 0.5). Which
settings over-smooth, which leave the noise?

---
## 2 · J-invariance: learning to denoise from the noisy data alone

**The trick (Batson & Royer 2019, "Noise2Self").** Split the voxels into groups $J$ (for example: every voxel is its own
group). A function $f$ is **J-invariant** if its output on the voxels of $J$ never looks at the input on $J$:

$$f(x)_J \text{ depends only on } x_{J^c}\quad\text{for every group } J .$$

Now suppose the noise is zero-mean and **independent from voxel to voxel**, $x = y + n$. Then for a J-invariant $f$

$$\mathbb{E}\,\lVert f(x) - x\rVert^2 \;=\; \mathbb{E}\,\lVert f(x) - y\rVert^2 \;+\; \mathbb{E}\,\lVert n\rVert^2 .$$

*Why:* expand $\lVert f(x)-y-n\rVert^2$; the cross term $-2\,\mathbb{E}[(f(x)_j - y_j)\,n_j]$ vanishes at every voxel $j$,
because $f(x)_j$ was computed without $x_j$ and therefore without $n_j$, and $n_j$ has mean zero.

*In words:* comparing the output with the **noisy input** gives the true error plus a constant (the noise power).
Minimising the self-supervised loss therefore minimises the true error — no clean data needed. Without J-invariance the
cheapest way to match the noisy input is to *copy* it, noise and all: the identity map.

**Noise2Noise (Lehtinen *et al.* 2018)** uses the same algebra with two independent noisy copies: compare $f(\text{even})$
with *odd*: $\mathbb{E}\lVert f(x_{\text{even}}) - x_{\text{odd}}\rVert^2 = \mathbb{E}\lVert f(x_{\text{even}}) - y\rVert^2 + \text{const}$,
for **any** $f$, because the odd noise is independent of everything $f$ sees.

Let us use both ideas to tune the width σ of a Gaussian filter **without the answer key** — and then check against it.

In [ ]:
# ── 2.1 Tuning a filter without clean data: Noise2Self and Noise2Noise ──
def gaussian_j_invariant(x, sigma):
    """J-invariant ('donut') Gaussian filter: remove the centre voxel's own weight and renormalise."""
    delta = np.zeros((9, 9, 9), np.float32); delta[4, 4, 4] = 1
    w0 = ndimage.gaussian_filter(delta, sigma)[4, 4, 4]            # weight the filter gives a voxel's own value
    return (ndimage.gaussian_filter(x, sigma) - w0 * x) / (1 - w0)


sigmas = np.array([0.4, 0.6, 0.8, 1.0, 1.2, 1.5, 2.0, 2.5])
mse = lambda a, b: float(np.mean((a - b) ** 2))
curves = {k: [] for k in ["self (Gaussian)", "true (Gaussian)", "self (J-inv.)", "true (J-inv.)", "N2N", "true (N2N)"]}
for s in sigmas:
    g, gj, ge = ndimage.gaussian_filter(noisy, s), gaussian_j_invariant(noisy, s), ndimage.gaussian_filter(even_n, s)
    curves["self (Gaussian)"].append(mse(g, noisy)); curves["true (Gaussian)"].append(mse(g, truth))
    curves["self (J-inv.)"].append(mse(gj, noisy)); curves["true (J-inv.)"].append(mse(gj, truth))
    curves["N2N"].append(mse(ge, odd_n)); curves["true (N2N)"].append(mse(ge, truth))

fig, axes = plt.subplots(1, 3, figsize=(11, 3.1))
pairs = [("self (Gaussian)", "true (Gaussian)", "ordinary Gaussian: ‖f(x) − x‖²"),
         ("self (J-inv.)", "true (J-inv.)", "J-invariant Gaussian: ‖f(x) − x‖²"),
         ("N2N", "true (N2N)", "Noise2Noise: ‖f(even) − odd‖²")]
for ax, (k_self, k_true, title) in zip(axes, pairs):
    ax.plot(sigmas, curves[k_self], "o-", label="self-supervised loss (no clean data)")
    ax.plot(sigmas, curves[k_true], "s--", label="true loss vs answer key")
    ax.set_title(title, fontsize=9); ax.set_xlabel("filter width σ (voxels)"); ax.set_ylabel("mean squared error")
axes[0].legend(fontsize=7)
plt.tight_layout(); plt.show()

best = {k: sigmas[int(np.argmin(v))] for k, v in curves.items()}
print(f"best σ — ordinary Gaussian: self-supervised says {best['self (Gaussian)']}, truth says {best['true (Gaussian)']}")
print(f"best σ — J-invariant:       self-supervised says {best['self (J-inv.)']}, truth says {best['true (J-inv.)']}")
print(f"best σ — Noise2Noise:       self-supervised says {best['N2N']}, truth says {best['true (N2N)']}")
gap = np.array(curves["self (J-inv.)"]) - np.array(curves["true (J-inv.)"])
print(f"J-invariant: self − true = {gap.mean():.3f} ± {gap.std():.3f}  (theory: the noise variance = {NOISE_VAR:.3f})")
SIGMA_N2S = best["self (J-inv.)"]
assert abs(best["self (J-inv.)"] - best["true (J-inv.)"]) <= 0.21, "Noise2Self should pick (almost) the answer key's σ"

🔍 **What just happened?** For the ordinary Gaussian (left) the self-supervised loss keeps falling as σ → 0: copying the
input is the best way to "predict" it, so this loss would tell you not to filter at all. For the **J-invariant** version
(middle) the two curves are **parallel**: the self-supervised loss is the true loss shifted up by (almost exactly) the noise
variance, so it picks the same best σ, or a neighbouring value on our coarse grid (the curves are flat near their minimum)
— computed from the noisy tomogram alone. Noise2Noise (right) does the same with two
half-dose tomograms. This is how you can tune *any* denoiser on real data.

⚠️ The guarantee needs noise that is **independent between voxels**. Real tomograms violate this: reconstruction and the
CTF correlate the noise of neighbouring voxels, so a voxel's neighbours partly "know" its noise. Blind-spot methods
therefore often hide a larger region than a single voxel — keep this in mind for the next section.

---
## 3 · J-invariant volume shuffle: a blind-spot network for one noisy tomogram

A **blind-spot network** is a neural network that is J-invariant by construction, so it can be trained with
$\lVert f(x) - x\rVert^2$ on the very tomogram you want to denoise. Liu, Kassab, Xu & Ho (WACV 2025) build one for cryo-ET:

1. **First layer:** a 5×5×5 *centrally masked* convolution — the central 3×3×3 block of its kernel is zeroed, so the
   first features at voxel $p$ never see $p$ or its immediate neighbours.
2. **Body:** blocks whose spatial convolutions are **dilated by 3** (they combine voxels 3 apart).
3. **Volume unshuffle / shuffle (VU/VS):** a 3D "pixel unshuffle" that rearranges a $D\times H\times W\times C$ volume into
   $\tfrac{D}{v}\times\tfrac{H}{v}\times\tfrac{W}{v}\times Cv^3$, replacing pooling in a 2-level U-shaped network.
   Setting the shuffle factor $v = 3$ equal to the dilation $d = 3$ keeps the blind spot intact; the paper proves that the
   whole network stays J-invariant (its Supplement S2).

**Why "factor = dilation" matters — a picture in one line.** A 3³ convolution dilated by 3 only ever combines voxels whose
coordinates differ by multiples of 3. Cut the volume into the 27 interleaved sub-volumes
$\{p : p \equiv s \pmod 3\}$ — that is exactly what unshuffling by 3 does — and the dilated convolution becomes an *ordinary*
convolution on each sub-volume separately. The first layer's mask guarantees that every feature at $p$ is computed from
voxels at offsets with at least one coordinate equal to ±2, i.e. from **other** sub-volumes than $p$'s own. As long as
later layers never mix sub-volumes, nothing ever feeds $x_p$ back into the output at $p$.

Our toy keeps the masked 5³ first layer and the volume unshuffle with $v = d = 3$, and processes the 27 sub-volumes
separately with shared weights (two 3³ convolutions = dilation-3 convolutions at full resolution). ⚠️ We could not read
the paper's Supplement S2, so we do not know exactly how its VU layers avoid mixing sub-volumes; we use the safest
construction and **check numerically** — including what happens if sub-volumes *are* mixed.

In [ ]:
# ── 3.1 Masked convolution and 3D volume (un)shuffle ──
class MaskedConv3d(nn.Conv3d):
    """k×k×k convolution whose central hole×hole×hole block of weights is forced to zero."""

    def __init__(self, c_in, c_out, k=5, hole=3):
        super().__init__(c_in, c_out, k, padding=k // 2)
        mask = torch.ones(1, 1, k, k, k)
        a = (k - hole) // 2
        mask[..., a:a + hole, a:a + hole, a:a + hole] = 0
        self.register_buffer("mask", mask)

    def forward(self, x):
        return F.conv3d(x, self.weight * self.mask, self.bias, padding=self.padding)


def volume_unshuffle(x, v=3):
    """[B, C, D, H, W] → the v³ interleaved sub-volumes, stacked along the batch: [B·v³, C, D/v, H/v, W/v]."""
    B, C, D, H, W = x.shape
    x = x.view(B, C, D // v, v, H // v, v, W // v, v).permute(0, 3, 5, 7, 1, 2, 4, 6)
    return x.reshape(B * v ** 3, C, D // v, H // v, W // v)


def volume_shuffle(x, B, v=3):
    """Inverse of :func:`volume_unshuffle`: [B·v³, C, d, h, w] → [B, C, d·v, h·v, w·v]."""
    _, C, d, h, w = x.shape
    x = x.view(B, v, v, v, C, d, h, w).permute(0, 4, 5, 1, 6, 2, 7, 3)
    return x.reshape(B, C, d * v, h * v, w * v)


class VolumeShuffleBSN(nn.Module):
    """Tiny J-invariant volume-shuffle network (illustration of Liu et al. 2025's design principle).

    masked=False gives an ordinary network (no blind spot); mix=True unshuffles the sub-volumes into CHANNELS and lets the
    next layers mix them — a deliberately broken variant used for the blind-spot check.
    """

    def __init__(self, ch=8, masked=True, mix=False):
        super().__init__()
        self.first = MaskedConv3d(1, ch) if masked else nn.Conv3d(1, ch, 5, padding=2)
        self.mix = mix
        c_body = ch * 27 if mix else ch
        self.body = nn.Sequential(nn.Conv3d(c_body, ch, 3, padding=1), nn.ReLU(),
                                  nn.Conv3d(ch, ch, 3, padding=1), nn.ReLU(),
                                  nn.Conv3d(ch, 27 if mix else 1, 1))

    def forward(self, x):
        B = x.shape[0]
        h = volume_unshuffle(F.relu(self.first(x)))                  # [B·27, ch, D/3, H/3, W/3]
        if self.mix:                                                  # sub-volumes → channels: they get mixed
            _, C, d, hh, w = h.shape
            out = self.body(h.reshape(B, 27 * C, d, hh, w))           # [B, 27, d, h, w]
            return volume_shuffle(out.reshape(B * 27, 1, d, hh, w), B)
        return volume_shuffle(self.body(h), B)                         # each sub-volume processed on its own

In [ ]:
# ── 3.2 The blind-spot check: does an output voxel depend on its own input? ──
def self_dependence(net, size=24, p=(12, 12, 12)):
    """|∂ output[p] / ∂ input[p]| and the full gradient map of output[p] w.r.t. the input (one backward pass)."""
    x = torch.randn(1, 1, size, size, size, requires_grad=True)
    net(x)[0, 0][p].backward()
    g = x.grad[0, 0]
    return float(g[p].abs()), g.abs().numpy()


torch.manual_seed(SEED)
checks = {"blind-spot (ours)": VolumeShuffleBSN(), "ordinary CNN": VolumeShuffleBSN(masked=False),
          "sub-volumes mixed": VolumeShuffleBSN(mix=True)}
grad_maps = {}
for name, net in checks.items():
    d_self, grad_maps[name] = self_dependence(net)
    print(f"{name:20s}: |∂out[p]/∂in[p]| = {d_self:.2e}   (voxels that influence out[p]: {(grad_maps[name] > 0).sum()})")
assert self_dependence(checks["blind-spot (ours)"])[0] == 0.0, "the blind-spot network must ignore its own input voxel"

fig, axes = plt.subplots(1, 3, figsize=(9, 3.0))
for ax, (name, gmap) in zip(axes, grad_maps.items()):
    sl = (gmap[12, 4:21, 4:21] > 0).astype(float)
    ax.imshow(sl, cmap="viridis", vmin=0, vmax=1, interpolation="nearest", extent=(-8.5, 8.5, 8.5, -8.5))
    ax.plot(0, 0, "r+", ms=10, mew=2)
    ax.set_title(f"{name}\n(z-offset 0 slice; yellow = seen)", fontsize=8); ax.set_xlabel("x offset (voxels)")
axes[0].set_ylabel("y offset (voxels)")
plt.tight_layout(); plt.show()

🔍 **What just happened?** We asked autograd how much the output at voxel $p$ (red cross) changes when the input at $p$
changes. For our volume-shuffle network the answer is **exactly zero**; for an ordinary CNN, and for the variant that mixes
sub-volumes in its channels, it is not. The left map shows the blind-spot network's receptive field: it sees almost
everything around $p$ except a regular **lattice of holes** — every voxel whose offset from $p$ is a multiple of 3 in all
three directions. That lattice is the network's "J": the volume is split into 27 interleaved groups, and each output voxel
is predicted only from the *other* 26.

In [ ]:
# ── 3.3 Train on ONE noisy volume (48³ crop): blind-spot vs ordinary network ──
CROP = (slice(0, 48), slice(24, 72), slice(24, 72))
V_noisy, V_truth = noisy[CROP], truth[CROP]


def train_on_one_volume(net, vol, steps=200, crop=24, batch=2, lr=3e-3, seed=SEED):
    """Self-supervised training ‖f(x) − x‖² on random crops of a single noisy volume; returns the denoised volume."""
    net = net.to(DEVICE)
    opt = torch.optim.Adam(net.parameters(), lr=lr)
    g = np.random.default_rng(seed)
    X = torch.from_numpy(vol)[None, None].to(DEVICE)
    t0 = time.time()
    for _ in range(steps):
        starts = [[g.integers(0, n - crop + 1) for n in vol.shape] for _ in range(batch)]
        xb = torch.cat([X[..., z:z + crop, y:y + crop, x:x + crop] for z, y, x in starts])
        loss = ((net(xb) - xb) ** 2).mean()
        opt.zero_grad(); loss.backward(); opt.step()
    with torch.no_grad():
        out = net(X)[0, 0].cpu().numpy()
    print(f"  {steps} steps in {time.time() - t0:.0f} s; final self-supervised loss {loss.item():.3f}")
    return out


torch.manual_seed(SEED)
print("blind-spot network:")
den_bsn = train_on_one_volume(VolumeShuffleBSN(), V_noisy)
torch.manual_seed(SEED)
print("ordinary network (same architecture, no mask):")
den_cnn = train_on_one_volume(VolumeShuffleBSN(masked=False), V_noisy)
den_gauss = gaussian_j_invariant(V_noisy, SIGMA_N2S)       # the Noise2Self-tuned filter from Section 2
den_gauss_plain = ndimage.gaussian_filter(V_noisy, SIGMA_N2S)

rows = [("noisy input", V_noisy), (f"Gaussian σ={SIGMA_N2S} (N2S-tuned)", den_gauss_plain),
        ("ordinary CNN, self-supervised", den_cnn), ("blind-spot network", den_bsn)]
for name, v in rows:
    print(f"{name:32s} PSNR {psnr(v, V_truth):6.2f} dB   correlation with its own input {pearson(v, V_noisy):.3f}")
fig, axes = plt.subplots(1, 5, figsize=(12, 2.7))
for ax, (name, v) in zip(axes, rows):
    ax.imshow(v[24]); ax.set_title(f"{name}\n{psnr(v, V_truth):.1f} dB", fontsize=8); ax.axis("off")
axes[-1].imshow(V_truth[24]); axes[-1].set_title("noise-free answer key", fontsize=8); axes[-1].axis("off")
plt.tight_layout(); plt.show()
assert psnr(den_bsn, V_truth) > psnr(V_noisy, V_truth) + 5, "the blind-spot network should denoise"
assert psnr(den_cnn, V_truth) < psnr(den_bsn, V_truth), "the ordinary network should be worse (identity collapse)"

🔍 **What just happened?** Both networks were trained the same way, on the same single noisy volume, with the same loss
$\lVert f(x)-x\rVert^2$. The **ordinary network** learned to copy its input (correlation with the input ≈ 1, PSNR ≈ the
noisy input) — the identity collapse predicted in Section 2. The **blind-spot network** cannot copy, so the only way to
lower its loss is to predict each voxel from its surroundings, which removes the independent noise: it gains many dB
without ever seeing clean data, matching or beating the best Gaussian (whose σ we tuned with Noise2Self). With more
channels, depth and training it learns structure-aware filters that a Gaussian cannot express.

✏️ **Your turn.** (a) Train longer (`steps=600`) or wider (`VolumeShuffleBSN(ch=16)`). (b) Make the noise *correlated*:
replace `V_noisy` by `ndimage.gaussian_filter(V_noisy, 0.7)` (neighbouring voxels now share noise) and retrain — what
happens to the blind-spot network's advantage?

### 📄 Paper connection — J-invariant volume shuffle

**Liu, Kassab, Xu & Ho (2025).** *J-Invariant Volume Shuffle for Self-Supervised Cryo-Electron Tomogram Denoising on Single
Noisy Volume.* WACV 2025, pp. 568–577. [arXiv:2411.15248](https://arxiv.org/abs/2411.15248)
- **Question it answers:** can we denoise a cryo-ET volume when the only data is that one noisy volume (no clean targets,
  no second noisy copy)?
- **What the paper does:** a J-invariant blind-spot network: a 5³ sparse centrally masked convolution (central 3³ zeroed),
  then Dilated Channel Attention blocks (LayerNorm, 1×1×1 conv, dilation-3 3³ depth-wise conv, SimpleGate, simplified
  channel attention, skip) following D-BSN; volume unshuffle/shuffle with $v = d = 3$ replaces pooling in a 2-level U-shape.
  Loss $\mathcal L=\lambda_1\lVert f(V)-V\rVert^2+\lambda_2\lVert f(V)-V^{f}\rVert^2+\lambda_3\lVert K(f(V))-K(V^{b})\rVert^2+\lambda_4\,\mathrm{TV}(f(V))$
  with $(\lambda_1..\lambda_4)=(0.8, 0.5, 0.05, 0.01)$: the self-supervised term, a Gaussian-guided term ($V^f$ = Gaussian-smoothed
  volume), an edge term ($K$ = Kirsch-type edge operator, $V^b$ = bilateral-filtered volume) and total variation.
- **What our toy keeps:** masked 5³ first layer with a 3³ hole; volume unshuffle with factor 3 = dilation 3; self-supervised
  training on a single volume; the numerical J-invariance check.
- **What it leaves out:** DCA blocks, the U-shape, the three extra loss terms, real data, and the paper's proof (we
  separate sub-volumes explicitly instead).
- **Result reported:** on real data (*G. hansenii*, vesicle, phage T4, centriole; IMOD reconstructions) the authors report
  the best even/odd FSC resolution at the 0.5 threshold on all four sets, e.g. 24.61 Å vs 27.95 Å (NMSG) on *G. hansenii*;
  on simulated SHREC 2021 data (σ = 0.1) 39.23 dB vs 36.47 dB for NMSG.
- **Code:** <https://github.com/Xiwei-web/SelfCryoET> (from the paper; not part of AITom).

---
## 4 · Validating without ground truth: even/odd FSC

With real data there is no answer key. The standard proxy (Module 10) splits the recorded electrons into two independent
halves — even and odd frames — reconstructs and denoises each half **separately**, and measures their **Fourier shell
correlation**:

$$\mathrm{FSC}(k) = \frac{\sum_{|\mathbf{k}|\in\text{shell}} F_{\text{even}}(\mathbf{k})\,F^{*}_{\text{odd}}(\mathbf{k})}
{\sqrt{\sum |F_{\text{even}}|^2\,\sum |F_{\text{odd}}|^2}} .$$

*In words:* at each spatial frequency, how consistent are the two independent halves? The noise is different in the two
halves, the signal is the same, so consistency means signal. Liu *et al.* report the frequency where FSC-e/o drops below 0.5.

💡 **A trap worth knowing.** A *linear, rotation-symmetric* filter (like a Gaussian) multiplies $F_{\text{even}}$ and
$F_{\text{odd}}$ by the **same** number on each shell — and that number cancels between numerator and denominator. Such a
filter makes the picture smoother but **cannot change FSC-e/o**. Let us check, and see what a learned, nonlinear denoiser does.

In [ ]:
# ── 4.1 Even/odd FSC: raw halves, Gaussian-filtered halves, network-denoised halves ──
E_crop, O_crop = even_n[CROP], odd_n[CROP]
bsn_for_halves = VolumeShuffleBSN()
torch.manual_seed(SEED)
print("training a blind-spot network on the EVEN half only:")
_ = train_on_one_volume(bsn_for_halves, E_crop)
with torch.no_grad():
    run = lambda v: bsn_for_halves(torch.from_numpy(v)[None, None].to(DEVICE))[0, 0].cpu().numpy()
    den_E, den_O = run(E_crop), run(O_crop)

pairs = {"raw halves": (E_crop, O_crop),
         f"Gaussian σ={SIGMA_N2S} on both": (ndimage.gaussian_filter(E_crop, SIGMA_N2S), ndimage.gaussian_filter(O_crop, SIGMA_N2S)),
         "blind-spot network on both": (den_E, den_O)}
fig, ax = plt.subplots(figsize=(6.2, 3.4))
for (name, (a, b)), style in zip(pairs.items(), ["-", "--", "-"]):
    f, c, _ = fsc(a, b, pixel_A=10.0)
    ax.plot(f, c, style, label=f"{name}: 0.5 at {resolution_at(f, c, 0.5):.0f} Å")
ax.axhline(0.5, color="gray", ls=":", lw=1)
ax.set_xlabel("spatial frequency (1/Å)"); ax.set_ylabel("FSC even vs odd"); ax.legend(fontsize=7)
ax.set_title("FSC-e/o: what a linear filter cannot change")
plt.tight_layout(); plt.show()
for name, (a, b) in pairs.items():
    print(f"{name:28s}: PSNR of the even half vs answer key {psnr(a, V_truth):5.1f} dB")

🔍 **What just happened?** The Gaussian curve lies on top of the raw curve (apart from small wiggles at the highest
frequencies, where the filter has removed almost everything and round-off dominates): **filtering made the halves prettier
but not more consistent**. FSC-e/o can only rise if a denoiser treats signal and noise *differently* — which a learned nonlinear
denoiser can do (it sees structure across many voxels) — and its rise is evidence that the denoiser exploited real,
reproducible signal. The flip side: a denoiser that invents the *same* structure in both halves (Section 7) would also
raise FSC-e/o. Use it together with other checks, never alone.

---
## 5 · DUAL: learn to simulate and to denoise from *unpaired* data

Supervised denoisers need pairs (noisy, clean). Experiments give us many **noisy tomograms**; maps and simulations give us
many **clean densities** — but never of the *same* object. DUAL (Zeng *et al.*, bioRxiv 2024) learns two directions at
once from these unpaired sets, in a CycleGAN-like framework:

- a **denoiser** $D_n$: noisy tomogram → clean density;
- a **simulator** = $G_s$ (structure-related "noise": missing wedge, defocus/CTF, aberration) + $G_n$ (non-structural noise
  generated from a random code $z \sim \mathcal{N}(0, I)$, then voxel-permuted), and an encoder $E$ that extracts the noise
  code from real tomograms;
- **discriminators** in real space **and in Fourier (spectral) space**, one per domain, that judge whether a simulated
  tomogram looks real and whether a denoised volume looks clean.

Loss: $L = \lambda_{rec}L_{rec} + \lambda_{adv}L_{adv} + \lambda_z L_z$ — cycle reconstruction (MSE + Pearson correlation),
adversarial (GAN) and a term pushing $E(\cdot)$ towards $\mathcal{N}(0, I)$.

In [ ]:
# ── 5.1 The DUAL idea at a glance (diagram) ──
fig, ax = plt.subplots(figsize=(9.2, 3.4))
B = dict(boxstyle="round,pad=0.35", fc="#eef3fb", ec="#4a6fa5")
D_ = dict(boxstyle="round,pad=0.3", fc="#fff6e0", ec="#c08a00")
R_ = dict(boxstyle="round,pad=0.25", fc="#fde8e8", ec="#c0392b")
ax.text(0.8, 2.6, "real noisy\ntomograms", ha="center", va="center", bbox=D_, fontsize=9)
ax.text(0.8, 0.6, "clean density\nmaps", ha="center", va="center", bbox=D_, fontsize=9)
ax.text(4.0, 2.6, "denoiser D_n", ha="center", va="center", bbox=B, fontsize=9)
ax.text(4.0, 0.6, "simulator G_s + G_n(z)\n(CTF, wedge, noise)", ha="center", va="center", bbox=B, fontsize=8)
ax.text(7.2, 2.6, "denoised\n(should look clean)", ha="center", va="center", bbox=D_, fontsize=8)
ax.text(7.2, 0.6, "simulated tomogram\n(should look real)", ha="center", va="center", bbox=D_, fontsize=8)
for (a, b) in [((1.55, 2.6), (3.3, 2.6)), ((5.0, 2.6), (6.3, 2.6)), ((1.5, 0.6), (2.9, 0.6)), ((5.2, 0.6), (6.1, 0.6))]:
    ax.annotate("", xy=b, xytext=a, arrowprops=dict(arrowstyle="->", lw=1.3))
ax.annotate("", xy=(4.4, 2.3), xytext=(7.0, 0.95), arrowprops=dict(arrowstyle="->", lw=1.2, color="#8e44ad",
                                                                 connectionstyle="arc3,rad=0.2"))
ax.text(5.9, 1.6, "cycle: D_n(simulated) ≈ clean", fontsize=8, color="#8e44ad")
ax.text(8.9, 2.6, "discriminators\nreal + Fourier\nspace", ha="center", va="center", bbox=R_, fontsize=8)
ax.text(8.9, 0.6, "discriminators\nreal + Fourier\nspace", ha="center", va="center", bbox=R_, fontsize=8)
for yy in (2.6, 0.6):
    ax.plot([8.05, 8.25], [yy, yy], color="k", lw=1)
ax.set_xlim(0, 9.8); ax.set_ylim(0, 3.3); ax.axis("off")
ax.set_title("DUAL (schematic): two unpaired domains, two directions, cycle + adversarial losses", fontsize=10)
plt.show()

### 5.2 Toy: why a cycle alone is not enough

We build two **unpaired** sets of 32×32 images: *clean* projections of zoo molecules, and *noisy* images of **other**
particles seen through a "microscope" (a Gaussian blur standing in for CTF and wedge, then strong noise). All maps are
linear and isotropic so that we can read them off:

- simulator: $G(c) = h * c + s\,\varepsilon$ — a learnable radial transfer function $h(|k|)$ (12 numbers) and noise level $s$;
- denoiser: $D_n(x) = d * x$ — another learnable radial filter $d(|k|)$.

Loss **(a)** cycle only: $\lVert D_n(G(c)) - c\rVert^2$. Loss **(b)** adds a cheap stand-in for DUAL's *Fourier-space
discriminators*: the radially averaged log power spectrum of $G(c)$ must match that of the real noisy set, and that of
$D_n(x)$ must match that of the clean set. ⚠️ This is an **illustration of the cycle + distribution-matching idea, not
DUAL's networks or losses** (no GAN, no U-Nets, no encoder $E$).

In [ ]:
# ── 5.2 Two unpaired domains ──
def random_projections(n, rng, size=32):
    """2D projections (sum along z) of randomly rotated zoo molecules, scaled to unit-ish intensity."""
    ids = list(templates)
    return np.stack([rotate_volume(templates[ids[rng.integers(len(ids))]], random_rotation(rng)).sum(0) / 8.0
                     for _ in range(n)]).astype(np.float32)


g5 = np.random.default_rng(SEED + 5)
t0 = time.time()
clean_set = random_projections(250, g5)                     # domain 1: clean maps
hidden_truth = random_projections(250, g5)                  # domain 2's (hidden) clean content — different particles
BLUR = 1.5
blurred = ndimage.gaussian_filter(hidden_truth, (0, BLUR, BLUR))
NOISE_SD_TOY = float(blurred.std() / np.sqrt(0.5))
noisy_set = blurred + g5.normal(0, NOISE_SD_TOY, blurred.shape).astype(np.float32)   # domain 2: "real" noisy data
print(f"made 2 × 250 unpaired images in {time.time() - t0:.1f} s; true noise σ = {NOISE_SD_TOY:.3f}")

In [ ]:
# ── 5.3 Cycle only vs cycle + spectral matching ──
K_GRID = np.sqrt(sum(k ** 2 for k in np.meshgrid(np.fft.fftfreq(32), np.fft.fftfreq(32), indexing="ij")))
N_BINS = 12
KB = torch.from_numpy(np.minimum(K_GRID / 0.5 * (N_BINS - 1), N_BINS - 1)).float()
SHELL = torch.from_numpy(np.minimum((K_GRID * 32).astype(int), 15)).long()


def radial_filter(img, profile):
    """Filter images [N, 32, 32] with a rotation-symmetric transfer function given at N_BINS radii (linear interpolation)."""
    i0 = KB.floor().long().clamp(max=N_BINS - 1); i1 = (i0 + 1).clamp(max=N_BINS - 1); w = KB - i0
    H = profile[i0] * (1 - w) + profile[i1] * w
    return torch.fft.ifft2(torch.fft.fft2(img) * H).real


def log_power_spectrum(img):
    """Radially averaged log power spectrum of a batch of images (16 shells)."""
    P = (torch.fft.fft2(img).abs() ** 2).mean(0).flatten()
    rad = torch.zeros(16).index_add(0, SHELL.flatten(), P) / torch.bincount(SHELL.flatten(), minlength=16)
    return torch.log(rad + 1e-8)


C_t, X_t, Y_t = map(torch.from_numpy, (clean_set, noisy_set, hidden_truth))
PS_NOISY, PS_CLEAN = log_power_spectrum(X_t), log_power_spectrum(C_t)


def train_dual_toy(spectral, steps=400, seed=SEED):
    """Learn simulator (h, s) and denoiser (d) from unpaired sets; returns (h, s, d)."""
    torch.manual_seed(seed)
    h, d = torch.ones(N_BINS, requires_grad=True), torch.ones(N_BINS, requires_grad=True)
    log_s = torch.tensor(np.log(0.05), dtype=torch.float32, requires_grad=True)
    opt = torch.optim.Adam([h, d, log_s], lr=0.02)
    gen = torch.Generator().manual_seed(seed)
    for _ in range(steps):
        c = C_t[torch.randint(0, len(C_t), (32,), generator=gen)]
        x = X_t[torch.randint(0, len(X_t), (32,), generator=gen)]
        simulated = radial_filter(c, h) + log_s.exp() * torch.randn(c.shape, generator=gen)
        loss = ((radial_filter(simulated, d) - c) ** 2).mean()                       # clean → noisy → clean
        if spectral:                                                                  # stand-in for Fourier discriminators
            loss = loss + 0.1 * ((log_power_spectrum(simulated) - PS_NOISY) ** 2).mean() \
                        + 0.1 * ((log_power_spectrum(radial_filter(x, d)) - PS_CLEAN) ** 2).mean()
        opt.zero_grad(); loss.backward(); opt.step()
    return h.detach(), log_s.exp().item(), d.detach()


t0 = time.time()
toy = {name: train_dual_toy(spectral) for name, spectral in [("cycle only", False), ("cycle + spectra", True)]}
print(f"trained both variants in {time.time() - t0:.0f} s")
mse_raw = float(((X_t - Y_t) ** 2).mean())
for name, (h, s, d) in toy.items():
    den = radial_filter(X_t, d)
    print(f"{name:16s}: simulator noise σ = {s:.3f} (true {NOISE_SD_TOY:.3f});  denoised MSE on the REAL noisy set "
          f"{float(((den - Y_t) ** 2).mean()):.4f} (raw {mse_raw:.4f})")
triv = float(((radial_filter(radial_filter(C_t, torch.ones(N_BINS)), torch.ones(N_BINS)) - C_t) ** 2).mean())
print(f"for reference: identity simulator (no blur, no noise) + identity denoiser gives a cycle error of {triv:.1e}")

fig, axes = plt.subplots(1, 6, figsize=(11, 2.2))
with torch.no_grad():
    sims = {n: radial_filter(C_t[:1], h) + s * torch.randn(1, 32, 32, generator=torch.Generator().manual_seed(1))
            for n, (h, s, d) in toy.items()}
panels = [("clean map", C_t[0]), ("sim. (cycle only)", sims["cycle only"][0]), ("sim. (+ spectra)", sims["cycle + spectra"][0]),
          ("REAL noisy", X_t[0]), ("denoised (+ spectra)", radial_filter(X_t[:1], toy["cycle + spectra"][2])[0]),
          ("hidden truth", Y_t[0])]
for ax, (t, im) in zip(axes, panels):
    ax.imshow(im.numpy()); ax.set_title(t, fontsize=8); ax.axis("off")
plt.tight_layout(); plt.show()

🔍 **What just happened?** The cycle loss can be driven to (almost) zero by a **useless simulator** — one that adds almost
no noise — because a clean image that is barely changed is easy to recover: the "identity simulator + identity
denoiser" pair has zero cycle error. Nothing in the cycle says what *real* data look like. Adding the spectral term forces
the simulator's output to have the power spectrum of the real noisy set: it learns a realistic noise level, its simulated
images look like the real ones, and the denoiser trained through the cycle on these realistic simulations works better on
the real data. In DUAL this role is played by adversarial discriminators in real and Fourier space, which match the whole
distribution, not just the power spectrum.

### 📄 Paper connection — DUAL

**Zeng, Ding, Zhang, Uddin, Dabouei & Xu (2024).** *DUAL: deep unsupervised simultaneous simulation and denoising for
cryo-electron tomography.* bioRxiv, [doi:10.1101/2024.03.02.583135](https://doi.org/10.1101/2024.03.02.583135)
(preprint; no journal version recorded in the bioRxiv metadata we checked; first names not listed there).
- **Question it answers:** can we get both a denoiser and a realistic simulator from unpaired experimental tomograms and
  clean density maps?
- **What the paper does:** a cycle-consistent, CycleGAN-like framework with U-Net generators: denoiser $D_n$, structural
  "noise" generator $G_s$ (missing wedge, CTF/defocus, spherical aberration), non-structural noise generator $G_n$ from a
  code $z\sim\mathcal N(0,I)$ (then voxel-permuted), encoder $E$; one discriminator per domain in real and Fourier space;
  $L=\lambda_{rec}L_{rec}+\lambda_{adv}L_{adv}+\lambda_zL_z$ (MSE + Pearson cycle loss; GAN loss; $E$'s codes towards
  $\mathcal N(0,I)$). A missing-wedge extension treats a tomogram and its axis-transposed copy as two domains (reported
  comparable to IsoNet). The λ values and $K$ were not available to us.
- **What our toy keeps:** unpaired domains; simulator + denoiser trained through a clean → noisy → clean cycle; a
  Fourier-space distribution constraint.
- **What it leaves out:** GANs, U-Nets, the encoder $E$ and the noisy → clean → noisy cycle, real-space discriminators, 3D,
  missing wedge, voxel permutation.
- **Result reported:** on SHREC 2021, SSIM rises from 0.011 (raw) to 0.568 with all baselines below 0.1; ribosome picking
  F1 rises from 0.577 (raw) to 0.702 (baselines 0.495–0.586).
- **Code:** the authors state it "will be available" through AITom; no DUAL folder existed in the AITom snapshot we used.

---
## 6 · Score-based denoising: learn what clean data look like

### 6.1 The score and Tweedie's formula

Let $y$ be a noisy observation of a clean value $x$ with Gaussian noise of variance $\sigma^2$, and let $p(y)$ be the
probability density of *noisy* observations. The **score** is the slope of its logarithm, $\nabla_y \log p(y)$ — an arrow
pointing towards more probable (more typical) observations. **Tweedie's formula** (Robbins 1956; Efron 2011) gives the best
possible denoiser in the least-squares sense from the score alone:

$$\mathbb{E}[x \mid y] \;=\; y + \sigma^2\,\nabla_y \log p(y).$$

*In words:* start at the noisy value and step uphill on the density of noisy data, by an amount set by the noise variance.
The remarkable part: the formula only needs the distribution of **noisy** data — which we have plenty of.

Check it in 1D: a voxel is either background ($x = 0$, probability 0.6) or protein ($x = 1$, probability 0.4), observed with
noise $\sigma = 0.5$. Here everything can be computed exactly.

In [ ]:
# ── 6.1 Tweedie's formula in 1D: score-based estimate vs exact Bayes ──
SIG1, P_BG = 0.5, 0.6
gauss_pdf = lambda y, m, s: np.exp(-(y - m) ** 2 / (2 * s ** 2)) / np.sqrt(2 * np.pi * s ** 2)
yg = np.linspace(-1.5, 2.5, 401)
p0, p1 = P_BG * gauss_pdf(yg, 0, SIG1), (1 - P_BG) * gauss_pdf(yg, 1, SIG1)
score = (p0 * (-yg / SIG1 ** 2) + p1 * (-(yg - 1) / SIG1 ** 2)) / (p0 + p1)      # d/dy log p(y)
tweedie = yg + SIG1 ** 2 * score
bayes = p1 / (p0 + p1)                                                            # E[x|y] = P(protein | y)
print(f"max |Tweedie − exact posterior mean| = {np.abs(tweedie - bayes).max():.1e}")
assert np.allclose(tweedie, bayes)

# Denoising score matching (DSM) in 1D: learn the score from noisy samples only
g6 = torch.Generator().manual_seed(SEED)
x_true = (torch.rand(20000, 1, generator=g6) > P_BG).float()
y_samples = x_true + SIG1 * torch.randn(20000, 1, generator=g6)                  # all we observe
SIG_A1 = 0.1
net1d = nn.Sequential(nn.Linear(1, 64), nn.SiLU(), nn.Linear(64, 64), nn.SiLU(), nn.Linear(64, 1))
opt = torch.optim.Adam(net1d.parameters(), lr=3e-3)
torch.manual_seed(SEED)
for step in range(1500):
    yb = y_samples[torch.randint(0, 20000, (512,), generator=g6)]
    eps = torch.randn(yb.shape, generator=g6)
    loss = ((SIG_A1 * net1d(yb + SIG_A1 * eps) + eps) ** 2).mean()                # ‖σ_a s(y + σ_a ε) + ε‖²
    opt.zero_grad(); loss.backward(); opt.step()
with torch.no_grad():
    learned = net1d(torch.from_numpy(yg).float()[:, None])[:, 0].numpy()

fig, axes = plt.subplots(1, 3, figsize=(11, 3.0))
axes[0].plot(yg, p0 + p1); axes[0].set_title("density of NOISY values p(y)"); axes[0].set_xlabel("y")
axes[1].plot(yg, score, label="exact score"); axes[1].plot(yg, learned, "--", label="learned by DSM (noisy data only)")
axes[1].set_title("score ∇log p(y)"); axes[1].set_xlabel("y"); axes[1].legend(fontsize=7)
axes[2].plot(yg, yg, ":", color="gray", label="no denoising")
axes[2].plot(yg, bayes, label="exact E[x | y]"); axes[2].plot(yg, yg + SIG1 ** 2 * learned, "--", label="Tweedie with learned score")
axes[2].set_title("denoised value"); axes[2].set_xlabel("noisy value y"); axes[2].legend(fontsize=7)
plt.tight_layout(); plt.show()

🔍 **What just happened?** The score (middle) is positive where moving right makes $y$ more probable and negative
otherwise; Tweedie turns it into the exact Bayesian posterior mean (right) — noisy values near 0 are pulled to background,
near 1 to protein. The dashed curves were learned by **denoising score matching (DSM)** from the noisy samples alone:

$$\mathcal{L}_{\text{DSM}} = \mathbb{E}\,\big\lVert \sigma_a\, s_\theta(y + \sigma_a\varepsilon) + \varepsilon \big\rVert^2,
\qquad \varepsilon\sim\mathcal{N}(0, I).$$

*In words:* add a little extra noise $\sigma_a\varepsilon$ to the data and train $s_\theta$ to point back along $-\varepsilon$;
the best such $s_\theta$ is the score of the (slightly smoothed) noisy distribution (Vincent 2011). Training on noisy data and
then applying Tweedie is the "Noise2Score" recipe (Kim & Ye 2021) — and the starting point of the lab's TSM code.

### 6.2 Score matching on particle images

Now images: 24×24 projections of zoo molecules in random orientations (≈ 1.3 nm pixels), with strong white noise
($\sigma = 0.5$ for particles of peak intensity ≈ 1) — a toy "particle stack" from single-particle cryo-EM. A small CNN
$s_\theta$ is trained with DSM ($\sigma_a = 0.2$, as in the AITom TSM configuration) on the **noisy** images only, then
denoises with one Tweedie step $\hat x = y + \sigma^2 s_\theta(y)$.

In [ ]:
# ── 6.2 A toy particle stack and a small score network ──
P_SIZE, SIG_IMG, SIG_A = 24, 0.5, 0.2


def particle_stack(n, rng):
    """Noise-free 24×24 projections of randomly rotated zoo molecules (cast of this notebook)."""
    ids = list(templates)
    return np.stack([ndimage.zoom(rotate_volume(templates[ids[rng.integers(len(ids))]], random_rotation(rng)).sum(0),
                                  P_SIZE / 32, order=1) for _ in range(n)]).astype(np.float32)


t0 = time.time()
g6 = np.random.default_rng(SEED + 6)
stack_clean = particle_stack(600, g6)
SCALE = float(np.median(stack_clean.max((1, 2))))
stack_clean /= SCALE
test_clean = particle_stack(150, g6) / SCALE
bank = particle_stack(60, np.random.default_rng(SEED + 60)) / SCALE        # clean references (other orientations)
stack_noisy = stack_clean + g6.normal(0, SIG_IMG, stack_clean.shape).astype(np.float32)
test_noisy = test_clean + g6.normal(0, SIG_IMG, test_clean.shape).astype(np.float32)
X_STACK = torch.from_numpy(stack_noisy)[:, None].to(DEVICE)
X_TESTP = torch.from_numpy(test_noisy)[:, None].to(DEVICE)
BANK = torch.from_numpy(bank)[:, None].to(DEVICE)
RANGE_P = float(test_clean.max() - test_clean.min())
psnr_p = lambda est: float(10 * np.log10(RANGE_P ** 2 / np.mean((est - test_clean) ** 2)))
print(f"particle stacks ready in {time.time() - t0:.0f} s: train {stack_noisy.shape}, test {test_noisy.shape}, bank {bank.shape}")


class ScoreNet(nn.Module):
    """Small CNN s_θ(x) ≈ ∇ log p(x); ``features`` exposes a middle layer (like the U-Net middle block used by TSM)."""

    def __init__(self, ch=16):
        super().__init__()
        self.inp = nn.Sequential(nn.Conv2d(1, ch, 3, padding=1), nn.SiLU(), nn.Conv2d(ch, ch, 3, padding=1), nn.SiLU())
        self.middle = nn.Sequential(nn.Conv2d(ch, ch, 3, padding=2, dilation=2), nn.SiLU())
        self.out = nn.Sequential(nn.Conv2d(ch, ch, 3, padding=1), nn.SiLU(), nn.Conv2d(ch, 1, 3, padding=1))

    def features(self, x):
        return self.middle(self.inp(x))

    def forward(self, x):
        return self.out(self.features(x))

In [ ]:
# ── 6.3 Train with denoising score matching (DSM) only ──
def edge_mask(x):
    """'Structure mask' of the TSM code: pixels whose gradient magnitude is above the batch mean."""
    gx = F.pad((x[..., :, 1:] - x[..., :, :-1]).abs(), (0, 1, 0, 0))
    gy = F.pad((x[..., 1:, :] - x[..., :-1, :]).abs(), (0, 0, 0, 1))
    g = gx + gy
    return (g > g.mean()).float()


def standardise(t):
    """Per-image zero mean / unit std (the code's norm(·)): keeps the spatial pattern, drops scale and offset."""
    return (t - t.mean((2, 3), keepdim=True)) / (t.std((2, 3), keepdim=True) + 1e-6)


@torch.no_grad()
def bank_features(model):
    """Mean-pooled middle-layer features of every clean reference, L2-normalised: [K, C]."""
    return F.normalize(model.features(BANK).mean((2, 3)), dim=1)


@torch.no_grad()
def soft_target(model, x, fb, tau=0.5):
    """TSM's clean target: softmax(cosine similarity / τ)-weighted average of the clean bank, using mean-pooled
    middle-layer features (as the code's CleanTargetProjector). ``fb`` = bank_features(model).
    Returns (target images, best similarity per image)."""
    fx = F.normalize(model.features(x).mean((2, 3)), dim=1)
    sims = fx @ fb.T                                                   # [B, K] cosine similarities
    weights = torch.softmax(sims / tau, dim=1)
    return torch.einsum("bk,kchw->bchw", weights, BANK), sims.max(1).values


def train_score(alpha_tsm=0.0, tsm_sign=1.0, steps=500, batch=32, seed=SEED, refresh=20):
    """DSM (+ optional target-guided term) on the noisy stack. Loss = L_DSM + α·λ(t)·mean(w·L_TSM).

    Bank features are recomputed every ``refresh`` steps (the code's projector also offers a refresh).
    """
    torch.manual_seed(seed)
    model = ScoreNet().to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=2e-3)
    gen = torch.Generator().manual_seed(seed)
    t0, gates = time.time(), []
    for step in range(steps):
        y = X_STACK[torch.randint(0, len(X_STACK), (batch,), generator=gen)]
        eps = torch.randn(y.shape, generator=gen).to(DEVICE)
        y_tilde = y + SIG_A * eps
        s = model(y_tilde)
        loss = ((SIG_A * s + eps) ** 2).mean()                                    # L_DSM
        if alpha_tsm > 0:
            lam = 0.0 if step < 50 else min(1.0, (step - 50) / 30)               # warm-up as in the code (50 + 30 steps)
            if step % refresh == 0:
                fb = bank_features(model)
            y_star, best_sim = soft_target(model, y, fb)                          # similarity computed on the noisy input
            w = torch.sigmoid(3.0 * (best_sim - 0.93))                            # gate: guide only bank-like patches
            gates.append(w.mean().item())
            direction = tsm_sign * (y_star - y_tilde)                             # from the input towards the soft target
            l_tsm = ((standardise(s) - standardise(direction)) ** 2 * edge_mask(y_tilde)).mean((1, 2, 3))
            loss = loss + alpha_tsm * lam * (w * l_tsm).mean()
        opt.zero_grad(); loss.backward(); opt.step()
    print(f"  {steps} steps in {time.time() - t0:.0f} s" + (f"; mean gate w = {np.mean(gates):.2f}" if gates else ""))
    return model.eval()


@torch.no_grad()
def tweedie(model, y, sigma=SIG_IMG):
    """One Tweedie step: x̂ = y + σ² s_θ(y)."""
    return (y + sigma ** 2 * model(y))[:, 0].cpu().numpy()


print("DSM only:")
score_dsm = train_score(alpha_tsm=0.0)
den_dsm = tweedie(score_dsm, X_TESTP)
best_gauss = max((psnr_p(ndimage.gaussian_filter(test_noisy, (0, s, s))), s) for s in [0.8, 1.0, 1.2, 1.5, 2.0])
print(f"PSNR on held-out particles: noisy {psnr_p(test_noisy):.2f} dB | best Gaussian (σ={best_gauss[1]}, chosen with "
      f"the answer key) {best_gauss[0]:.2f} dB | DSM + Tweedie {psnr_p(den_dsm):.2f} dB")

### 6.4 Target guidance (TSM): borrow a bank of clean references

In single-particle cryo-EM we often *do* have some clean data: projections of an earlier map, or of an ab initio
reconstruction. The lab's **"Score-Based Matching with Target Guidance for Cryo-EM Denoising"** (ECCV 2026) uses such a
**bank of clean targets** to steer score matching. ⚠️ We could not find the paper's text; what follows is read from the
**released AITom code** (`aitom/filter/denoise/tsm`):

1. **Soft clean target.** Features of a noisy patch (mean-pooled activations of the network's middle block) are compared
   by cosine similarity with the features of every clean reference $c_k$; the target is
   $y^\star = \sum_k \mathrm{softmax}_k(\mathrm{sim}_k/\tau)\,c_k$ with $\tau = 0.5$.
2. **Guidance term.** On a "structure mask" (pixels with above-average gradient), the *spatial pattern* of the predicted
   score — after per-patch standardisation $\mathrm{norm}(\cdot)$, which removes scale — is matched to the pattern of the
   direction connecting the noisy input and the soft target.
3. **Gate.** Each patch is weighted by $w = \mathrm{sigmoid}(3(\mathrm{sim}_{\max} - 0.93))$, so only patches that resemble
   some clean reference are guided.
4. **Total loss:** $\mathcal{L} = \mathcal{L}_{\text{DSM}} + \alpha\,\lambda(t)\,w\,\mathcal{L}_{\text{TSM}}$, with
   $\lambda(t)$ ramping from 0 to 1 after a warm-up; inference uses repeated Tweedie-like steps.

Our toy keeps all four pieces with a 60-image clean bank (projections of the same molecules in *other* random
orientations), the code's default weight $\alpha = 0.08$, and guidance pointing **from the noisy input towards the
target** ($y^\star - \tilde y$), the direction in which Tweedie's formula moves an image (see the paper box for a note on
the sign used in the released code). ⚠️ This tiny toy is **not a fair test** of the published method (different network,
data scale and bank); it lets you see the mechanism at work.

In [ ]:
# ── 6.4 Train with DSM + target guidance and compare ──
ALPHA_TSM, TSM_SIGN = 0.08, +1.0                   # the code's default weight; our sign convention (see box)
print("DSM + target guidance:")
score_tsm = train_score(alpha_tsm=ALPHA_TSM, tsm_sign=TSM_SIGN)
den_tsm = tweedie(score_tsm, X_TESTP)
_, sim_test = soft_target(score_tsm, X_TESTP, bank_features(score_tsm))
gate_test = torch.sigmoid(3.0 * (sim_test - 0.93)).cpu().numpy()
per_img = lambda est: 10 * np.log10(RANGE_P ** 2 / np.mean((est - test_clean) ** 2, axis=(1, 2)))
gain = per_img(den_tsm) - per_img(den_dsm)
hi = gate_test >= np.median(gate_test)
print(f"PSNR on held-out particles: DSM {psnr_p(den_dsm):.2f} dB  vs  DSM + TSM {psnr_p(den_tsm):.2f} dB")
print(f"per-image gain of guidance: images with a high gate (bank-like) {gain[hi].mean():+.2f} dB, "
      f"low gate {gain[~hi].mean():+.2f} dB")

with torch.no_grad():
    y_star_test, _ = soft_target(score_tsm, X_TESTP, bank_features(score_tsm))
y_star_test = y_star_test[:, 0].cpu().numpy()
fig, axes = plt.subplots(5, 6, figsize=(8.4, 7.0))
for j in range(6):
    for r, (name, im) in enumerate([("noisy", test_noisy), ("DSM + Tweedie", den_dsm), ("DSM + TSM", den_tsm),
                                    ("soft target y* (what guidance pulls towards)", y_star_test), ("answer key", test_clean)]):
        axes[r, j].imshow(im[j], vmin=-0.3, vmax=1.2); axes[r, j].axis("off")
        if j == 0:
            axes[r, j].set_title(name, fontsize=8, loc="left")
plt.tight_layout(); plt.show()

🔍 **What just happened?** *(Read the printed numbers — they can differ slightly on your machine.)* A score network
trained **only on noisy particles** beats the best Gaussian filter, even though we chose the Gaussian's width by peeking at
the answer key. With the code's small guidance weight the guided network behaves almost like the unguided one here. Look
at the fourth row: the **soft targets** are blurred, orientation-averaged particles — our bank shows the molecules in other
orientations, and mean-pooled features cannot tell orientations apart. Guidance pulls the learned prior towards such
targets; whether that helps depends entirely on how well the bank matches the data.

✏️ **Your turn.** (a) Set `ALPHA_TSM = 1.0` and re-run 6.4: in this toy the PSNR *drops*, because the network is pulled
towards the blurred targets — a lesson about mismatched references. (b) Make the problem harder (`SIG_IMG = 1.0` in 6.2,
re-run 6.2–6.4): is a strong prior more useful when the data are noisier? (c) Flip `TSM_SIGN = -1.0` to use the released
code's literal target direction and compare.

### 📄 Paper connection — Score-based matching with target guidance (TSM)

**"Score-Based Matching with Target Guidance for Cryo-EM Denoising."** ECCV 2026 (as listed in AITom's publication list).
> ⚠️ We could not access the paper's text (or verify its author list) while writing; the description below is limited to
> the released code and README. Read the paper before relying on details.
- **Question it answers (from the README):** denoise 2D cryo-EM micrographs/particle images without clean pairs, using a
  bank of clean target images (e.g. projections at 10 Å and 3 Å, or ab initio maps) to steer the denoiser.
- **What the code does:** a guided-diffusion-style 2D U-Net outputs a score; **DSM on noisy data** (Noise2Score style)
  $\lVert\sigma_a s_\theta(\tilde y)+\varepsilon\rVert^2$ with $\sigma_a = 0.2$; **target-guided term**: soft target
  $y^\star=\sum_k\mathrm{softmax}(\mathrm{sim}_k/0.5)\,c_k$ from mean-pooled middle-block features (a 10 Å / 3 Å two-bank
  "mix" mode averages two softmaxes), input $x'=y/\sqrt{\sigma^2+\sigma_a^2}+\sigma_a\varepsilon$, loss
  $\frac{\sigma^2\sigma_a^2+\sigma^4}{\sigma^2}\lVert \mathrm{norm}(\tfrac{\sigma_a}{\sigma\sqrt{\sigma_a^2+\sigma^2}}s_\theta(x'))-\mathrm{norm}(\tfrac{x'-y^\star/\sqrt{\sigma_a^2+\sigma^2}}{\sigma})\rVert^2$
  on a structure mask; gate $w=\mathrm{sigmoid}(3(\mathrm{sim}_{\max}-0.93))$; total $\mathcal L_{DSM}+\alpha\lambda(t)w\mathcal L_{TSM}$
  with $\alpha = 0.08$ and a warm-up of 50 + 30 steps. **Inference:** five steps $x\leftarrow x+\eta\,\sigma(x)^2 s_\theta(x)$
  with a signal-dependent $\sigma(x)=0.3+0.01x$ and $\eta=0.7$.
- **A critical-reading note (ours).** Because of the per-patch standardisation, all positive constants inside
  $\mathrm{norm}(\cdot)$ cancel and the term matches spatial *patterns*. In the code the DSM term trains the network output
  to point along $-\varepsilon$ (from noisy towards clean), while the target pattern is written $x' - y^\star/\sqrt{\cdot}$,
  i.e. from the target towards the input. Taken literally the two terms pull in opposite directions; without the paper we
  cannot tell whether a different sign convention is intended. Our toy uses the Tweedie-consistent direction (exercise (c)
  lets you test the literal one).
- **What our toy keeps:** DSM on noisy data, Tweedie denoising, soft target from feature similarity with τ = 0.5, gate,
  structure mask, per-patch standardisation, warm-up.
- **What it leaves out:** the U-Net, micrograph-scale patches and sliding-window inference, the 10 Å/3 Å mix, the input
  rescaling by $1/\sqrt{\sigma^2+\sigma_a^2}$, the signal-dependent noise model, σ_a schedules.
- **Result reported:** not verified (no paper text). EMPIAR IDs 10081, 10291 and 11183 appear only as README examples.
- **Code:** [`aitom/filter/denoise/tsm`](https://github.com/xulabs/aitom/tree/master/aitom/filter/denoise/tsm).

---
## 7 · ⚠️ Hallucination: when the prior paints the picture

Every denoiser that beats a plain filter does so by using a **prior** — knowledge of what clean data usually look like.
The prior is also its weakness: it reports what it *expects*. Two small experiments show how.

### 7.1 The prior snaps to what it knows

Back to the 1D voxel of Section 6.1, whose prior allows only background (0) or full protein density (1). Suppose the real
voxel is **half occupied** (true value 0.5 — say a ligand bound in half of the copies), observed many times with noise
$\sigma = 0.25$. What does the Bayes-optimal (Tweedie) denoiser report?

In [ ]:
# ── 7.1 A half-occupied voxel through a {0, 1} prior ──
SIG7 = 0.25
y7 = 0.5 + SIG7 * np.random.default_rng(SEED + 7).standard_normal(5000)
q0, q1 = P_BG * gauss_pdf(y7, 0, SIG7), (1 - P_BG) * gauss_pdf(y7, 1, SIG7)
estimate = q1 / (q0 + q1)                                        # = y + σ² ∇log p(y), Tweedie with the {0,1} prior
fig, ax = plt.subplots(figsize=(5.6, 2.8))
ax.hist(y7, bins=60, alpha=0.5, label="noisy observations")
ax.hist(estimate, bins=60, alpha=0.7, label="denoised (Tweedie, prior {0, 1})")
ax.axvline(0.5, color="k", ls="--", label="true value 0.5")
ax.set_xlabel("voxel value"); ax.set_ylabel("count"); ax.legend(fontsize=7)
ax.set_title("A denoiser reports what its prior allows")
plt.tight_layout(); plt.show()
print(f"denoised values within 0.1 of the truth: {np.mean(np.abs(estimate - 0.5) < 0.1):.0%};  "
      f"snapped below 0.1 or above 0.9: {np.mean((estimate < 0.1) | (estimate > 0.9)):.0%}")

🔍 **What just happened?** The raw observations are centred on the truth (0.5), but the denoised values pile up near 0
and 1: each observation is confidently "explained" as background or as full protein — a state the prior knows — and the
real, intermediate state almost disappears. A sharper prior (less noise, or a network trained on narrower data) snaps
harder. Partially occupied sites, rare conformations and anything absent from the training data are exactly what such a
prior hides.

### 7.2 Guidance can pull empty regions towards particles

Target guidance (Section 6.4) decides *which* patches to guide with the gate $w = \mathrm{sigmoid}(3(\mathrm{sim}_{\max}-0.93))$
computed from feature similarity to the clean bank. What does it do with patches that contain **no particle at all**?

In [ ]:
# ── 7.2 Gates and soft targets for empty (noise-only) patches ──
empty = SIG_IMG * torch.randn(150, 1, P_SIZE, P_SIZE, generator=torch.Generator().manual_seed(7)).to(DEVICE)
fb = bank_features(score_tsm)
y_star_empty, sim_empty = soft_target(score_tsm, empty, fb)
_, sim_part = soft_target(score_tsm, X_TESTP, fb)
gate = lambda sim: torch.sigmoid(3.0 * (sim - 0.93)).cpu().numpy()
print(f"best feature similarity to the bank: particles {sim_part.mean():.3f}, empty patches {sim_empty.mean():.3f}")
print(f"mean gate w: particles {gate(sim_part).mean():.2f}, empty patches {gate(sim_empty).mean():.2f}")
fig, axes = plt.subplots(2, 6, figsize=(7.6, 2.8))
for j in range(6):
    axes[0, j].imshow(empty[j, 0].cpu()); axes[1, j].imshow(y_star_empty[j, 0].cpu(), vmin=-0.3, vmax=1.2)
    axes[0, j].axis("off"); axes[1, j].axis("off")
axes[0, 0].set_title("empty patch (pure noise)", fontsize=8, loc="left")
axes[1, 0].set_title("its soft target y* — a particle!", fontsize=8, loc="left")
plt.tight_layout(); plt.show()

🔍 **What just happened?** Mean-pooled features of pure noise resemble those of noisy particles, so empty patches get
gates comparable to real particles, and their "clean target" is — by construction — a blend of particles. Training on
such pairs teaches the network that an empty patch should be pulled towards a particle. This is the image-processing
cousin of "Einstein from noise" (Module 14): **a reference can reproduce itself from nothing**. The released code's gate
threshold (0.93) and structure mask are meant to limit this; whether they are sufficient on real micrographs is an
empirical question the paper (which we could not read) should answer.

**A checklist before trusting a denoised tomogram or micrograph:**
- Draw biological conclusions from **raw** (or linearly filtered) data, using the denoised volume as a guide.
- Check that features appear in **independent halves** (even/odd) *and* in raw data; FSC-e/o alone can be fooled by
  consistent hallucinations.
- Run the denoiser on **empty regions or pure noise** with the same statistics — anything that appears there is invented.
- Beware **reference banks and training sets** that contain the very structure you hope to find.
- Report the denoiser, its training data and its settings.

### 📄 Optional box — MicroFM: an optical cousin (fluorescence, not cryo-EM)

**Zhan, Jiang, Gupta, Swaminathan, Wang, Zhang, Wang & Xu (2026).** *MicroFM: Physics-guided Flow Matching for Isotropic
Microscopy Reconstruction.* CVPR 2026 (verified via the OpenReview record; first author **Xingzu** Zhan).
- **Not cryo-EM.** It addresses **3D fluorescence (light) microscopy**, where the axial (z) resolution is much worse than the
  lateral one because of an anisotropic point spread function (PSF) — the optical cousin of the missing wedge (Module 07).
- **What the abstract says:** training pairs are synthesised with **physical PSFs matched to the target microscope** (rather
  than a generic Gaussian blur), and a **physics-guided flow-matching** generative model restores isotropic resolution, with a
  continuous implicit geometry prior for volumetric consistency; state of the art on four fluorescence systems.
- ⚠️ We could not read the full paper: architecture, losses and numbers are unverified. For orientation only, the
  *generic* conditional flow-matching recipe (not paper-specific) is $x_t=(1-t)x_0+t x_1$ and
  $\mathcal L=\mathbb E\lVert v_\theta(x_t,t\mid y)-(x_1-x_0)\rVert^2$: learn a velocity field that carries noise $x_0$ to a
  clean image $x_1$, conditioned on the blurred observation $y$ — a close relative of the score models of Section 6.

---
## ✅ Checkpoint

1. Why does training an ordinary network with the loss $\lVert f(x) - x\rVert^2$ fail, and why does a blind-spot network not?
   <details><summary>Answer</summary>An ordinary network can lower the loss to zero by copying its input (identity), noise
   included. A J-invariant network cannot see $x_j$ when predicting voxel $j$, so the noise $n_j$ is unpredictable; the best it
   can do is predict the signal from the neighbourhood, and the loss equals the true error plus the noise variance.</details>
2. What assumption about the noise does Noise2Self need, and how do real tomograms violate it?
   <details><summary>Answer</summary>Zero-mean noise that is independent between voxels (between the groups J). Reconstruction
   (filtered back-projection) and the CTF correlate noise between neighbouring voxels, so neighbours leak information about a
   voxel's noise — one motivation for larger blind spots.</details>
3. Why must the unshuffle factor equal the dilation in the volume-shuffle network?
   <details><summary>Answer</summary>A 3³ convolution with dilation 3 only combines voxels whose offsets are multiples of 3 — i.e.
   voxels of the same interleaved sub-volume. Unshuffling by exactly 3 separates those sub-volumes, so the dilated convolutions
   become ordinary convolutions inside each one, and the blind spot created by the masked first layer is never filled in
   (as long as sub-volumes are not mixed, as our check showed).</details>
4. A Gaussian filter applied to both even and odd halves leaves FSC-e/o unchanged. Why?
   <details><summary>Answer</summary>It multiplies both halves' Fourier coefficients by the same real factor on each shell; the
   factor appears in the numerator and (squared, under a square root) in the denominator, and cancels.</details>
5. Write Tweedie's formula and explain each symbol.
   <details><summary>Answer</summary>$\mathbb E[x|y] = y + \sigma^2\nabla_y\log p(y)$: $y$ noisy observation, $x$ clean value,
   $\sigma^2$ noise variance, $p(y)$ density of noisy observations, $\nabla\log p$ its score; the posterior mean is a step uphill
   on the noisy-data density.</details>
6. In DUAL, what stops the simulator from producing unrealistic (e.g. noise-free) tomograms?
   <details><summary>Answer</summary>Not the cycle loss (it is happy with a near-identity simulator) but the discriminators in
   real and Fourier space, which require simulated tomograms to be indistinguishable from real ones. Our toy replaced them by a
   power-spectrum match.</details>

## 🧠 Questions & discussion

1. You denoise a tomogram and see a filament nobody has reported. List the checks you would run before telling your PI.
2. The volume-shuffle network is trained on the very tomogram it denoises. What are the advantages (no domain shift) and the
   risks (over-fitting to that volume, compute per tomogram)?
3. Target guidance uses clean references. When would that bias be acceptable (e.g. picking known particles) and when not
   (e.g. discovering a new conformation)?
4. DUAL learns a simulator as a by-product. How could a realistic simulator help the picking and classification methods of
   Modules 16–20 and 31?
5. PSNR rewards blur; SSIM rewards structure; FSC-e/o rewards consistency. Which would you report for a paper on membrane
   segmentation, and why?

## 🚀 Bonus — going further

- **Official code.** Run the volume-shuffle denoiser from <https://github.com/Xiwei-web/SelfCryoET> on a tomogram from the
  [CZ cryoET Data Portal](https://cryoetdataportal.czscience.com/) and compare with a Gaussian filter tuned by Noise2Self.
- **The paper's extra losses.** Add a Gaussian-guided term $\lVert f(V) - V^f\rVert^2$ and total variation to Section 3.3
  with the paper's weights (0.8, 0.5, 0.01) and see how the PSNR and the look change.
- **A real GAN.** Replace the spectral term of Section 5 by a small discriminator on images and on log power spectra.
- **TSM in AITom.** Read `aitom/filter/denoise/tsm/utils/utils.py::loss_func` and `CleanTargetProjector.py` and map each line to
  Section 6.4.
- **Deep dives** in this repository: [`../../LowSNRNoiseProcessing/lowdose_em_denoising_tutorial.ipynb`](../../LowSNRNoiseProcessing/lowdose_em_denoising_tutorial.ipynb)
  and [`../../../../sci_img/tutorial/SpatialPerceptionAnalysis/BioimageDenoising/`](../../../../sci_img/tutorial/SpatialPerceptionAnalysis/BioimageDenoising/).

## 📝 Lab note

```text
Module 24 — date:
Input: simulated tomogram (48×96×96, SNR 0.3, even/odd halves); toy particle stack (24×24, σ = 0.5)
Output: denoised volumes/images, PSNR/SSIM, FSC-e/o curves, hallucination test
Method: classical filters; Noise2Self/Noise2Noise tuning; blind-spot volume-shuffle net; DUAL-style cycle toy; DSM + Tweedie; TSM
Key parameter changed & effect:
Failure mode observed (identity collapse, correlated noise, hallucination):
Paper link: Liu et al. WACV 2025; Zeng et al. bioRxiv 2024; TSM (ECCV 2026, code)
```

## 📚 References

1. Liu X, Kassab M, Xu M, Ho Q. J-invariant volume shuffle for self-supervised cryo-electron tomogram denoising on single noisy volume. *WACV* 2025, pp. 568–577. arXiv:2411.15248.
2. Zeng X, Ding Y, Zhang Y, Uddin M, Dabouei A, Xu M. DUAL: deep unsupervised simultaneous simulation and denoising for cryo-electron tomography. bioRxiv (2024). doi:10.1101/2024.03.02.583135
3. Score-based matching with target guidance for cryo-EM denoising. *ECCV* 2026 (code: AITom `aitom/filter/denoise/tsm`). ⚠️ text and authors not verified.
4. Zhan X, Jiang R, Gupta V, Swaminathan T, Wang Y, Zhang G, Wang H, Xu M. MicroFM: physics-guided flow matching for isotropic microscopy reconstruction. *CVPR* 2026.
5. Batson J, Royer L. Noise2Self: blind denoising by self-supervision. *ICML* 2019. arXiv:1901.11365.
6. Lehtinen J *et al.* Noise2Noise: learning image restoration without clean data. *ICML* 2018. arXiv:1803.04189.
7. Vincent P. A connection between score matching and denoising autoencoders. *Neural Computation* 23:1661–1674 (2011).
8. Efron B. Tweedie's formula and selection bias. *J Am Stat Assoc* 106:1602–1614 (2011).
9. Kim K, Ye JC. Noise2Score: Tweedie's approach to self-supervised image denoising without clean images. *NeurIPS* 2021.
10. Zhu J-Y, Park T, Isola P, Efros AA. Unpaired image-to-image translation using cycle-consistent adversarial networks (CycleGAN). *ICCV* 2017.
11. Perona P, Malik J. Scale-space and edge detection using anisotropic diffusion. *IEEE TPAMI* 12:629–639 (1990).

## ➡️ Next

**Part F begins: [25 · Molecules as clouds of Gaussians](25_gaussian_mixture_molecules.ipynb)** — from averaging many
copies into one map to describing how each copy differs: Gaussian mixture models, neural networks and molecular motion.
Previous: [23 · Separating what a molecule is from how it is posed](23_disentangling_identity_pose.ipynb).
Course overview: [`README.md`](../README.md) · [`GLOSSARY.md`](../GLOSSARY.md) · [`PAPER_ATLAS.md`](../PAPER_ATLAS.md).